## Preprocessing

In [3]:
import findspark
findspark.init()

In [2]:

import argparse
import logging

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, FloatType, DoubleType
)
from pyspark.sql.window import Window

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

FANNIE_SCHEMA = StructType([
    StructField("reference_pool_id", StringType(), True),
    StructField("loan_id", StringType(), True),
    StructField("monthly_reporting_period", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("seller_name", StringType(), True),
    StructField("servicer_name", StringType(), True),
    StructField("master_servicer", StringType(), True),
    StructField("orig_interest_rate", DoubleType(), True),
    StructField("current_interest_rate", DoubleType(), True),
    StructField("orig_upb", DoubleType(), True),
    StructField("upb_at_issuance", DoubleType(), True),
    StructField("current_actual_upb", DoubleType(), True),
    StructField("orig_loan_term", IntegerType(), True),
    StructField("origination_date", StringType(), True),
    StructField("first_payment_date", StringType(), True),
    StructField("loan_age", IntegerType(), True),
    StructField("remaining_months_legal_mat", IntegerType(), True),
    StructField("remaining_months_to_mat", IntegerType(), True),
    StructField("maturity_date", StringType(), True),
    StructField("orig_ltv", FloatType(), True),
    StructField("orig_cltv", FloatType(), True),
    StructField("num_borrowers", IntegerType(), True),
    StructField("dti", FloatType(), True),
    StructField("borrower_credit_score_orig", IntegerType(), True),
    StructField("coborrower_credit_score_orig", IntegerType(), True),
    StructField("first_time_homebuyer", StringType(), True),
    StructField("loan_purpose", StringType(), True),
    StructField("property_type", StringType(), True),
    StructField("num_units", IntegerType(), True),
    StructField("occupancy_status", StringType(), True),
    StructField("property_state", StringType(), True),
    StructField("msa", StringType(), True),
    StructField("zip_short", StringType(), True),
    StructField("mi_percent", FloatType(), True),
    StructField("amortization_type", StringType(), True),
    StructField("prepayment_penalty", StringType(), True),
    StructField("io_flag", StringType(), True),
    StructField("io_first_pi_date", StringType(), True),
    StructField("months_to_amortization", IntegerType(), True),
    StructField("current_delinquency_status", StringType(), True),
    StructField("loan_payment_history", StringType(), True),
    StructField("modification_flag", StringType(), True),
    StructField("mi_cancellation_indicator", StringType(), True),
    StructField("zero_balance_code", StringType(), True),
    StructField("zero_balance_effective_date", StringType(), True),
    StructField("upb_at_removal", DoubleType(), True),
    StructField("repurchase_date", StringType(), True),
    StructField("scheduled_principal", DoubleType(), True),
    StructField("total_principal", DoubleType(), True),
    StructField("unscheduled_principal", DoubleType(), True),
    StructField("last_paid_installment_date", StringType(), True),
    StructField("foreclosure_date", StringType(), True),
    StructField("disposition_date", StringType(), True),
    StructField("foreclosure_costs", DoubleType(), True),
    StructField("property_preservation_costs", DoubleType(), True),
    StructField("asset_recovery_costs", DoubleType(), True),
    StructField("misc_holding_expenses", DoubleType(), True),
    StructField("taxes_holding", DoubleType(), True),
    StructField("net_sales_proceeds", DoubleType(), True),
    StructField("credit_enhancement_proceeds", DoubleType(), True),
    StructField("repurchase_make_whole_proceeds", DoubleType(), True),
    StructField("other_foreclosure_proceeds", DoubleType(), True),
    StructField("mod_non_interest_bearing_upb", DoubleType(), True),
    StructField("principal_forgiveness", DoubleType(), True),
    StructField("orig_list_start_date", StringType(), True),
    StructField("orig_list_price", DoubleType(), True),
    StructField("current_list_start_date", StringType(), True),
    StructField("current_list_price", DoubleType(), True),
    StructField("borrower_credit_score_issuance", IntegerType(), True),
    StructField("coborrower_credit_score_issuance", IntegerType(), True),
    StructField("borrower_credit_score_current", IntegerType(), True),
    StructField("coborrower_credit_score_current", IntegerType(), True),
    StructField("mi_type", StringType(), True),
    StructField("servicing_activity_indicator", StringType(), True),
    StructField("current_period_mod_loss", DoubleType(), True),
    StructField("cumulative_mod_loss", DoubleType(), True),
    StructField("current_period_credit_event_net", DoubleType(), True),
    StructField("cumulative_credit_event_net", DoubleType(), True),
    StructField("special_eligibility_program", StringType(), True),
    StructField("foreclosure_principal_writeoff", DoubleType(), True),
    StructField("relocation_mortgage", StringType(), True),
    StructField("zbc_change_date", StringType(), True),
    StructField("loan_holdback_indicator", StringType(), True),
    StructField("loan_holdback_effective_date", StringType(), True),
    StructField("delinquent_accrued_interest", DoubleType(), True),
    StructField("property_valuation_method", StringType(), True),
    StructField("high_balance_loan", StringType(), True),
    StructField("arm_init_fixed_le5y", StringType(), True),
    StructField("arm_product_type", StringType(), True),
    StructField("initial_fixed_rate_period", IntegerType(), True),
    StructField("interest_rate_adj_freq", IntegerType(), True),
    StructField("next_rate_adj_date", StringType(), True),
    StructField("next_payment_change_date", StringType(), True),
    StructField("index", StringType(), True),
    StructField("arm_cap_structure", StringType(), True),
    StructField("init_rate_cap_up", FloatType(), True),
    StructField("periodic_rate_cap_up", FloatType(), True),
    StructField("lifetime_rate_cap_up", FloatType(), True),
    StructField("mortgage_margin", FloatType(), True),
    StructField("arm_balloon", StringType(), True),
    StructField("arm_plan_number", IntegerType(), True),
    StructField("borrower_assistance_plan", StringType(), True),
    StructField("hltv_refi_option", StringType(), True),
    StructField("deal_name", StringType(), True),
    StructField("repurchase_make_whole_flag", StringType(), True),
    StructField("alt_delinquency_resolution", StringType(), True),
    StructField("alt_delinquency_res_count", IntegerType(), True),
    StructField("total_deferral_amount", DoubleType(), True),
    StructField("payment_deferral_mod_event", StringType(), True),
    StructField("interest_bearing_upb", DoubleType(), True),
    StructField("orig_classic_fico", IntegerType(), True),
    StructField("issuance_classic_fico", IntegerType(), True),
    StructField("current_classic_fico", IntegerType(), True),
])

# US states with mandatory judicial foreclosure (court approval at every step).
# Slower default process → subtle interaction with voluntary prepayment hazard.
_JUDICIAL_STATES = [
    "CT", "DE", "FL", "HI", "IL", "IN", "IA", "KS", "KY", "LA",
    "ME", "MD", "MA", "MN", "MO", "MT", "NE", "NJ", "NM", "NY",
    "ND", "OH", "OK", "PA", "RI", "SC", "SD", "VT", "WI",
]


# ===========================================================================
# SPARK SESSION
# ===========================================================================

def create_spark(app: str = "FannieMae_CPR",
                 driver_mem: str = "64g",
                 shuffle: int = 400) -> SparkSession:
    """
    Local SparkSession (default) for a single workstation.
    Driver gets 24 GB; 8 GB off-heap for large Parquet scans.
    """
    return (
        SparkSession.builder.appName(app)
        .config("spark.driver.memory",                         driver_mem)
        .config("spark.serializer",
                "org.apache.spark.serializer.KryoSerializer")
        .config("spark.kryoserializer.buffer.max",             "512m")
        .config("spark.sql.adaptive.enabled",                  "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled","true")
        .config("spark.sql.adaptive.skewJoin.enabled",         "true")
        .config("spark.sql.shuffle.partitions",        str(shuffle))
        .config("spark.sql.autoBroadcastJoinThreshold",        "100m")
        .config("spark.sql.parquet.filterPushdown",            "true")
        .config("spark.memory.offHeap.enabled",                "true")
        .config("spark.memory.offHeap.size",                   "8g")
        .getOrCreate()
    )


def create_spark_server(app: str = "FannieMae_CPR",
                        master: str = "spark://master:7077") -> SparkSession:
    """
    Distributed SparkSession for the actual cluster:
      - Master node  : 32 GB RAM, 8 CPU   →  driver gets 24 GB, 4 cores
      - 2 × datanode : 48 GB RAM, 12 CPU  →  each executor: 38 GB heap + 4 GB overhead

    Tuning rationale
    ----------------
    * spark.executor.instances=2    : one executor per 48-GB datanode.
    * spark.executor.cores=10       : leaves 2 cores per node for OS + HDFS DataNode.
    * spark.executor.memory=38g     : leaves ~6 GB per node for OS + daemon overhead.
    * spark.executor.memoryOverhead=4g : native/off-heap (total container = 42 GB).
    * spark.driver.memory=24g       : leaves ~8 GB on master for OS + Spark master daemon.
    * spark.driver.cores=4          : leaves 4 cores on master for OS + Spark master.
    * spark.sql.shuffle.partitions=200 : ~10× total executor cores (2 × 10 = 20).

    Usage
    -----
        python preprocessing/data_preprocessing.py \\
            --spark_mode server --master spark://master:7077 \\
            --data_path /data/raw --out_path /data/processed
    """
    return (
        SparkSession.builder.appName(app)
        .master(master)
        .config("spark.driver.memory",                         "24g")
        .config("spark.driver.cores",                          "4")
        .config("spark.executor.instances",                    "2")
        .config("spark.executor.cores",                        "10")
        .config("spark.executor.memory",                       "38g")
        .config("spark.executor.memoryOverhead",               "4g")
        .config("spark.serializer",
                "org.apache.spark.serializer.KryoSerializer")
        .config("spark.kryoserializer.buffer.max",             "512m")
        .config("spark.sql.adaptive.enabled",                  "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled","true")
        .config("spark.sql.adaptive.skewJoin.enabled",         "true")
        .config("spark.sql.shuffle.partitions",                "200")
        .config("spark.sql.autoBroadcastJoinThreshold",        "200m")
        .config("spark.sql.parquet.filterPushdown",            "true")
        .config("spark.memory.offHeap.enabled",                "true")
        .config("spark.memory.offHeap.size",                   "4g")
        .getOrCreate()
    )


# ===========================================================================
# LOAD CSV
# ===========================================================================

def load_raw(spark, path: str):
    log.info("Loading raw data: %s", path)
    df = (spark.read
          .option("sep", "|")
          .option("header", "false")
          .option("nullValue", "").schema(FANNIE_SCHEMA).csv(path))
    log.info("Raw rows in csv: {:,}".format(df.count()))
    return df


# ===========================================================================
# CLEANING AND LABELING
# ===========================================================================

def clean_and_label(df):
    # Best available FICO
    df = df.withColumn(
        "fico",
        F.coalesce(
            F.col("orig_classic_fico").cast("float"),
            F.col("borrower_credit_score_orig").cast("float")
        )
    )

    df = (
        df
        # Date parsing
        .withColumn("reporting_date", F.to_date("monthly_reporting_period", "MMyyyy"))
        .withColumn("origination_dt", F.to_date("origination_date", "MMyyyy"))
        .withColumn("first_payment_dt", F.to_date("first_payment_date", "MMyyyy"))
        .withColumn(
            "zbc_effective_dt",
            F.when(
                F.col("zero_balance_effective_date").isNotNull(),
                F.to_date("zero_balance_effective_date", "MMyyyy")
            )
        )
        .withColumn(
            "maturity_dt",
            F.when(
                F.col("maturity_date").isNull(),
                F.when(
                    F.col("first_payment_dt").isNotNull() &
                    F.col("orig_loan_term").isNotNull(),
                    F.add_months(
                        F.col("first_payment_dt"),
                        F.col("orig_loan_term") - 1
                    )
                )
            ).otherwise(
                F.to_date("maturity_date", "MMyyyy")
            )
        )
        .withColumn(
            "loan_age",
            F.coalesce(
                F.col("loan_age"),
                F.when(
                    F.col("first_payment_dt").isNotNull() & F.col("reporting_date").isNotNull(),
                    F.floor(F.months_between(F.col("reporting_date"), F.col("first_payment_dt"))) + 1
                ).cast("int")
            )
        )
        .withColumn(
            "current_actual_upb",
            F.greatest(
                F.coalesce(F.col("current_actual_upb"), F.lit(0.0)),
                F.coalesce(F.col("upb_at_removal"), F.lit(0.0))
            )
        )
        .withColumn(
            "current_actual_upb",
            F.when(
                F.col("current_actual_upb") <= 0,
                F.col("orig_upb")
            ).otherwise(F.col("current_actual_upb"))
        )

        # Vintages
        .withColumn("vintage_year", F.year("origination_dt"))
        .withColumn(
            "vintage_quarter",
            F.concat(
                F.year("origination_dt").cast("string"),
                F.lit("Q"),
                F.quarter("origination_dt").cast("string")
            )
        )

        # Fixed-rate mortgages only
        .filter(F.col("amortization_type") == "FRM")
        .filter(F.col("orig_loan_term").isin(180, 360))
        .filter(F.col("fico").isNotNull() & F.col("fico").between(300, 850))
        .filter(F.col("orig_ltv").isNotNull() & F.col("orig_ltv").between(1, 200))
        .filter(F.col("dti").isNotNull() & F.col("dti").between(1, 65))
        .filter(F.col("orig_interest_rate").between(1.0, 20.0))
        .filter(F.col("orig_upb") > 0)
        .filter(F.col("loan_age") >= 0)

        # Outcome labels
        .withColumn("prepaid", (F.col("zero_balance_code") == "01").cast("int"))
        .withColumn(
            "default",
            F.col("zero_balance_code")
             .isin("02", "03", "09", "15", "97", "98")
             .cast("int")
        )
        .withColumn(
            "removed",
            F.col("zero_balance_code")
             .isin("06", "16", "96")
             .cast("int")
        )
        .withColumn("active", F.col("zero_balance_code").isNull().cast("int"))

        # FICO tier
        .withColumn(
            "fico_bucket",
            F.when(F.col("fico") < 620, "SubPrime")
             .when(F.col("fico") < 680, "NearPrime")
             .when(F.col("fico") < 740, "Prime")
             .otherwise("SuperPrime")
        )

        # Binary structural flags
        .withColumn("high_ltv", (F.col("orig_ltv") > 80).cast("int"))
        .withColumn("term_15y", (F.col("orig_loan_term") <= 180).cast("int"))
        .withColumn(
            "is_refi",
            F.col("loan_purpose").isin("C", "R", "U").cast("int")
        )
        .withColumn("is_cashout", (F.col("loan_purpose") == "C").cast("int"))
        .withColumn("is_io", (F.col("io_flag") == "Y").cast("int"))
        .withColumn("has_ppm", (F.col("prepayment_penalty") == "Y").cast("int"))
        .withColumn("modified", (F.col("modification_flag") == "Y").cast("int"))
        .withColumn("is_investor", (F.col("occupancy_status") == "I").cast("int"))
        .withColumn("is_high_bal", (F.col("high_balance_loan") == "Y").cast("int"))
        .withColumn(
            "first_time_buyer",
            (F.col("first_time_homebuyer") == "Y").cast("int")
        )
        .withColumn(
            "in_forbearance",
            F.col("borrower_assistance_plan")
             .isin("F", "R", "T", "O")
             .cast("int")
        )
        .withColumn("has_deferral", (F.col("total_deferral_amount") > 0).cast("int"))

        # Spread / equity / burnout
        .withColumn(
            "rate_spread",
            F.col("orig_interest_rate") - F.col("current_interest_rate")
        )
        .withColumn(
            "equity_proxy",
            F.lit(1.0) - F.col("orig_ltv") / F.lit(100.0)
        )
        .withColumn(
            "delinquency_months",
            F.when(F.col("current_delinquency_status") == "XX", None)
             .otherwise(F.col("current_delinquency_status").cast("int"))
        )
        .withColumn(
            "seasoning_bucket",
            F.when(F.col("loan_age") <= 12, "0-12m")
             .when(F.col("loan_age") <= 36, "13-36m")
             .when(F.col("loan_age") <= 60, "37-60m")
             .when(F.col("loan_age") <= 120, "61-120m")
             .otherwise("120m+")
        )

        # Payment history features
        .withColumn(
            "ph_delinq_count",
            F.when(
                F.col("loan_payment_history").isNotNull(),
                (
                    F.length("loan_payment_history") / 2
                    - F.length(
                        F.regexp_replace("loan_payment_history", "00", "")
                    ) / 2
                ).cast("integer")
            ).otherwise(0)
        )
        .withColumn(
            "ph_recent_delinq",
            F.when(
                F.length("loan_payment_history") >= 6,
                (
                    F.regexp_replace(
                        F.substring("loan_payment_history", -6, 6),
                        "00",
                        ""
                    ) != ""
                ).cast("int")
            ).otherwise(0)
        )

        # Amortisation progress
        .withColumn(
            "upb_fraction",
            F.when(
                F.col("orig_upb") > 0,
                F.col("current_actual_upb") / F.col("orig_upb")
            )
        )
        .withColumn(
            "excess_principal",
            F.greatest(
                F.col("total_principal") - F.col("scheduled_principal"),
                F.lit(0.0)
            )
        )
        .withColumn(
            "burnout",
            F.col("loan_age") * F.greatest(F.col("rate_spread"), F.lit(0.0))
        )

        # ── NEW: Calendar month (seasonal prepayment patterns) ────────────
        # Prepayments peak in spring/summer (Mar–Jul) when homes change hands.
        # String encoding lets OHE create 12 dummies with no ordinal assumption.
        .withColumn("month_of_year", F.month("reporting_date").cast("string"))

        # ── NEW: Judicial foreclosure state ──────────────────────────────
        # Slower default competing-risk in these states may interact with
        # voluntary prepayment hazard.
        .withColumn(
            "is_judicial_state",
            F.when(
                F.col("property_state").isin(_JUDICIAL_STATES),
                1
            ).otherwise(0)
        )

        # ── NEW: HARP / High-LTV Refi Option ─────────────────────────────
        # hltv_refi_option == 'Y' marks loans eligible for Fannie Mae's
        # High LTV Refinance Option (HARP successor).  These borrowers
        # can refinance above 97% LTV, giving them unusually strong
        # prepayment incentive despite limited home equity.
        .withColumn(
            "is_hltv_refi",
            F.when(F.col("hltv_refi_option") == "Y", 1).otherwise(0)
        )
    )

    df = df.drop(
        "seller_name",
        "servicer_name",
        "master_servicer",
        "upb_at_issuance",
        "months_to_amortization",
        "io_first_pi_date",
        "arm_init_fixed_le5y",
        "arm_product_type",
        "initial_fixed_rate_period",
        "interest_rate_adj_freq",
        "next_rate_adj_date",
        "next_payment_change_date",
        "index",
        "arm_cap_structure",
        "init_rate_cap_up",
        "periodic_rate_cap_up",
        "lifetime_rate_cap_up",
        "mortgage_margin",
        "arm_balloon",
        "arm_plan_number",
        "deal_name",
        "current_period_mod_loss",
        "cumulative_mod_loss",
        "current_period_credit_event_net",
        "cumulative_credit_event_net",
        "orig_list_start_date",
        "orig_list_price",
        "current_list_start_date",
        "current_list_price",
        "borrower_credit_score_issuance",
        "coborrower_credit_score_issuance",
        "borrower_credit_score_current",
        "coborrower_credit_score_current"
    )

    # Forward-fill current_interest_rate within each loan (handles nulls
    # that arise when the servicer doesn't report the rate every period)
    w_rate = (
        Window.partitionBy("loan_id")
        .orderBy("reporting_date")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )

    df = df.withColumn(
        "current_interest_rate",
        F.coalesce(
            F.last("current_interest_rate", ignorenulls=True).over(w_rate),
            F.col("orig_interest_rate")
        )
    )

    log.info("Clean rows: {:,}".format(df.count()))
    return df


def build_abt(df):
    """One row per loan – latest reporting snapshot."""
    w = Window.partitionBy("loan_id").orderBy(F.desc("reporting_date"))
    abt = (df.withColumn("_rn", F.row_number().over(w))
             .filter(F.col("_rn") == 1).drop("_rn"))
    log.info("ABT rows (one per loan): {:,}".format(abt.count()))
    return abt


def save(df, path: str):
    log.info("Writing Parquet → %s", path)
    (df.write.mode("overwrite")
       .option("compression", "snappy")
       .parquet(path))
    log.info("Done.")


def main(data_path: str, out_path: str, spark_mode: str = "local",
         master: str = "spark://master:7077"):
    if spark_mode == "server":
        log.info("Starting server-mode SparkSession (64 GB master + 4×32 GB executors)")
        spark = create_spark_server(master=master)
    else:
        log.info("Starting local SparkSession")
        spark = create_spark()

    raw = load_raw(spark, data_path)
    cleaned = clean_and_label(raw)
    save(cleaned, out_path + "/panel")
    abt = build_abt(cleaned)
    save(abt, out_path + "/abt")
    cleaned.groupBy("zero_balance_code").count().orderBy("zero_balance_code").show()
    spark.stop()


In [8]:
!export SPARK_LOCAL_IP=$(hostname -I | awk '{print $1}')

IOStream.flush timed out


In [3]:
spark = create_spark()

26/04/14 21:52:13 WARN Utils: Your hostname, compute-vm-64-128-512-hdd-1776192960124 resolves to a loopback address: 127.0.1.1; using 10.130.0.13 instead (on interface eth0)
26/04/14 21:52:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 21:52:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [12]:
#spark.stop()

In [4]:
data_path = "/home/yc-user/raw_data"
out_path = "/home/yc-user/processed_data"

In [5]:
raw = load_raw(spark, data_path)

2026-04-14 21:52:22,222 [INFO] Loading raw data: /home/yc-user/raw_data
2026-04-14 21:52:27,019 [INFO] Raw rows in csv: 89,427,866                      


In [6]:
cleaned = clean_and_label(raw)

2026-04-14 21:53:26,892 [INFO] Clean rows: 82,244,050                           


In [7]:
save(cleaned, out_path + "/panel")

2026-04-14 21:53:52,682 [INFO] Writing Parquet → /home/yc-user/processed_data/panel
26/04/14 21:53:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
2026-04-14 21:56:27,196 [INFO] Done.                                            


In [8]:
abt = build_abt(cleaned)

2026-04-14 21:57:06,940 [INFO] ABT rows (one per loan): 1,293,420               


In [9]:
save(abt, out_path + "/abt")

2026-04-14 21:57:18,321 [INFO] Writing Parquet → /home/yc-user/processed_data/abt
2026-04-14 21:59:16,958 [INFO] Done.                                            


In [10]:
cleaned.groupBy("zero_balance_code").count().orderBy("zero_balance_code").show()

+-----------------+--------+
|zero_balance_code|   count|
+-----------------+--------+
|             NULL|81161135|
|               01| 1075043|
|               02|    1086|
|               03|     248|
|               06|    2360|
|               09|    1806|
|               15|     579|
|               16|    1793|
+-----------------+--------+



In [12]:
spark.stop()

## Preprocessing 2

In [1]:
import findspark
findspark.init()
import argparse
import json
import logging
import os
import pickle
import time
import gc

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import pyarrow.parquet as pq

# ── sklearn ──────────────────────────────────────────────────────────────────
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    brier_score_loss,
)
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import OneHotEncoder as SkOHE, StandardScaler

# ── Survival analysis ────────────────────────────────────────────────────────
from lifelines import CoxPHFitter, KaplanMeierFitter

# ── XGBoost ──────────────────────────────────────────────────────────────────
import xgboost as xgb

# ── LightGBM ────────────────────────────────────────────────────────────────
import lightgbm as lgb

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)
sns.set_theme(style="whitegrid", font_scale=1.1)


# ============================================================================
#  CONSTANTS  (shared with model_improvements_v3 and inference_pipeline_v3)
# ============================================================================

NUMERIC_FEATURES = [
    "fico", "orig_interest_rate", "current_interest_rate",
    "refi_incentive", "refi_incentive_pos",
    "rate_spread_to_10y", "spread_pos",
    "orig_ltv", "dti", "orig_upb", "upb_fraction", "equity_proxy",
    "loan_age", "age_sq", "burnout", "pct_term_elapsed",
    "orig_loan_term", "remaining_months_to_mat", "rate_duration",
    "burnout_x_refi", "fico_x_refi", "ltv_x_refi",
    "ph_delinq_count", "excess_principal", "gs10_monthly",
]

CATEGORICAL_FEATURES = [
    "channel", "loan_purpose", "property_type", "occupancy_status",
    "fico_bucket", "seasoning_bucket", "month_of_year", "vintage_year",
]

BINARY_FEATURES = [
    "high_ltv", "term_15y", "is_refi", "is_cashout", "is_io",
    "has_ppm", "modified", "is_investor", "is_high_bal",
    "first_time_buyer", "in_forbearance", "has_deferral",
    "is_judicial_state", "is_hltv_refi",
]

TARGET_COL     = "smm_target"
TRAIN_CUTOFF   = "2014-10-31"
SEED           = 42

_JUDICIAL_STATES = [
    "CT", "DE", "FL", "HI", "IL", "IN", "IA", "KS", "KY", "LA",
    "ME", "MD", "MA", "MN", "MO", "MT", "NE", "NJ", "NM", "NY",
    "ND", "OH", "OK", "PA", "RI", "SC", "SD", "VT", "WI",
]


# ============================================================================
#  PHASE 1:  SPARK PREPROCESSING  (feature engineering only, no encoding)
# ============================================================================
# Spark computes derived features that require window functions, joins,
# or UDFs.  The result is exported as raw scalar columns — no Spark ML
# Vectors, no OHE, no scaling.  All encoding is done in sklearn (Phase 2).
# ============================================================================

def create_spark(app="FannieMae_V3", driver_mem="64g"):
    """Local SparkSession for preprocessing only."""
    from pyspark.sql import SparkSession
    return (
        SparkSession.builder.appName(app)
        .config("spark.driver.memory",                          driver_mem)
        .config("spark.serializer",
                "org.apache.spark.serializer.KryoSerializer")
        .config("spark.kryoserializer.buffer.max",              "512m")
        .config("spark.sql.adaptive.enabled",                   "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled","true")
        .config("spark.sql.adaptive.skewJoin.enabled",          "true")
        .config("spark.sql.shuffle.partitions",                 "128")
        .config("spark.sql.autoBroadcastJoinThreshold",         "100m")
        .config("spark.sql.parquet.filterPushdown",             "true")
        .config("spark.memory.offHeap.enabled",                 "true")
        .config("spark.memory.offHeap.size",                    "8g")
        .getOrCreate()
    )


def create_spark_server(app="FannieMae_V3", master="yarn"):
    """Distributed SparkSession for Yandex DataProc YARN cluster."""
    from pyspark.sql import SparkSession
    return (
        SparkSession.builder.appName(app).master(master)
        .config("spark.submit.deployMode",                      "client")
        .config("spark.dynamicAllocation.enabled",              "false")
        .config("spark.driver.memory",                          "20g")
        .config("spark.driver.cores",                           "4")
        .config("spark.executor.instances",                     "2")
        .config("spark.executor.cores",                         "12")
        .config("spark.executor.memory",                        "33g")
        .config("spark.executor.memoryOverhead",                "4g")
        .config("spark.serializer",
                "org.apache.spark.serializer.KryoSerializer")
        .config("spark.kryoserializer.buffer.max",              "512m")
        .config("spark.sql.adaptive.enabled",                   "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled","true")
        .config("spark.sql.adaptive.skewJoin.enabled",          "true")
        .config("spark.sql.shuffle.partitions",                 "200")
        .config("spark.sql.autoBroadcastJoinThreshold",         "200m")
        .config("spark.sql.parquet.filterPushdown",             "true")
        .getOrCreate()
    )


def load_panel_spark(spark, panel_path: str, fred_path: str):
    """
    Load panel Parquet, build smm_target, join FRED GS10,
    and compute ALL derived features in Spark.

    Returns a Spark DataFrame with raw scalar columns
    (no Spark ML vectors, no OHE, no scaling).
    """
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    log.info("Loading panel Parquet from %s", panel_path)
    panel = spark.read.parquet(panel_path)
    log.info("Panel rows: {:,}".format(panel.count()))

    w = Window.partitionBy("loan_id").orderBy("reporting_date")
    df = (
        panel
        .withColumn("next_zbc", F.lead("zero_balance_code", 1).over(w))
        .withColumn(TARGET_COL, F.when(F.col("next_zbc") == "01", 1).otherwise(0))
        .filter(F.col("zero_balance_code").isNull())
        .filter(~F.col("current_delinquency_status")
                .isin("06", "07", "08", "09", "12", "XX"))
        .withColumn("upb_fraction",
            F.when(F.col("orig_upb") > 0,
                   F.col("current_actual_upb") / F.col("orig_upb")).otherwise(1.0))
        .withColumn("pct_term_elapsed",
            F.when(F.col("orig_loan_term") > 0,
                   F.col("loan_age") / F.col("orig_loan_term")).otherwise(0.0))
        .withColumn("excess_principal",
            F.greatest(F.col("total_principal") - F.col("scheduled_principal"), F.lit(0.0)))
        .withColumn("in_forbearance",
            F.when(F.isnan("in_forbearance") | F.col("in_forbearance").isNull(), 0)
             .otherwise(F.col("in_forbearance")))
        .withColumn("has_deferral",
            F.when(F.isnan("has_deferral") | F.col("has_deferral").isNull(), 0)
             .otherwise(F.col("has_deferral")))
        .withColumn("has_ppm",
            F.when(F.isnan("has_ppm") | F.col("has_ppm").isNull(), 1)
             .otherwise(F.col("has_ppm")))
        .withColumn("remaining_months_to_mat",
            F.coalesce(F.col("remaining_months_to_mat"),
                F.greatest(F.floor(F.months_between(F.col("maturity_dt"),
                    F.col("reporting_date"))), F.lit(0)).cast("int")))
    )

    # ── Derived features ──────────────────────────────────────────────────────
    df = (df
        .withColumn("refi_incentive",
            F.col("orig_interest_rate") - F.col("current_interest_rate"))
        .withColumn("refi_incentive_pos",
            F.greatest(F.col("orig_interest_rate") - F.col("current_interest_rate"), F.lit(0.0)))
        .withColumn("burnout",
            F.col("loan_age") * F.greatest(
                F.col("orig_interest_rate") - F.col("current_interest_rate"), F.lit(0.0)))
        .withColumn("month_of_year", F.month("reporting_date").cast("string"))
        .withColumn("is_judicial_state",
            F.col("property_state").isin(_JUDICIAL_STATES).cast("int"))
        .withColumn("is_hltv_refi",
            F.when(F.col("hltv_refi_option") == "Y", 1).otherwise(0))
    )

    # ── FRED GS10 join ────────────────────────────────────────────────────────
    log.info("Loading FRED GS10 from %s", fred_path)
    gs10 = (
        spark.read.option("header", "true").option("inferSchema", "false").csv(fred_path)
        .withColumn("month_date", F.trunc(F.to_date("observation_date", "yyyy-MM-dd"), "month"))
        .withColumn("gs10_monthly", F.col("GS10").cast("double"))
        .select("month_date", "gs10_monthly")
        .filter(F.col("month_date").isNotNull())
    )
    df = (df
        .withColumn("month_date", F.trunc("reporting_date", "month"))
        .join(F.broadcast(gs10), on="month_date", how="left")
        .withColumn("rate_spread_to_10y",
            F.when(F.col("current_interest_rate").isNotNull() &
                   F.col("gs10_monthly").isNotNull(),
                   F.col("current_interest_rate") - F.col("gs10_monthly")))
    )

    # ── Additional derived features ───────────────────────────────────────────
    df = (df
        .withColumn("spread_pos", F.greatest(F.col("rate_spread_to_10y"), F.lit(0.0)))
        .withColumn("age_sq", F.col("loan_age") * F.col("loan_age") / F.lit(100.0))
        .withColumn("rate_duration",
            F.when(F.col("remaining_months_to_mat").isNotNull() &
                   F.col("current_interest_rate").isNotNull(),
                   F.col("current_interest_rate") * F.col("remaining_months_to_mat") / F.lit(1200.0))
             .otherwise(F.lit(0.0)))
        .withColumn("burnout_x_refi",
            F.col("burnout") * F.col("refi_incentive_pos"))
        .withColumn("fico_x_refi",
            F.when(F.col("fico").isNotNull(),
                   F.col("fico") / F.lit(100.0) * F.col("refi_incentive_pos"))
             .otherwise(F.lit(0.0)))
        .withColumn("ltv_x_refi",
            F.when(F.col("orig_ltv").isNotNull(),
                   F.col("orig_ltv") * F.col("refi_incentive_pos"))
             .otherwise(F.lit(0.0)))
    )

    if "vintage_year" not in df.columns:
        df = df.withColumn("vintage_year", F.year("origination_dt").cast("string"))

    log.info("Panel with target rows: {:,}".format(df.count()))
    return df


def export_raw_data(df, out_path: str):
    """
    Export RAW scalar columns (no encoding, no scaling) to Parquet.

    Unlike V2 which used Spark ML Pipeline for encoding, V3 exports the
    raw feature columns.  All encoding/imputation/scaling happens in
    sklearn (Phase 2), so the sklearn objects are directly reusable at
    inference time.
    """
    from pyspark.sql import functions as F

    num_feats = [c for c in NUMERIC_FEATURES if c in df.columns]
    cat_feats = [c for c in CATEGORICAL_FEATURES if c in df.columns]
    bin_feats = [c for c in BINARY_FEATURES if c in df.columns]

    select_cols = (
        [TARGET_COL, "reporting_date", "origination_dt", "loan_id"]
        + num_feats + cat_feats + bin_feats
    )
    # Extra columns for Cox / KM
    for c in ["loan_age", "fico_bucket"]:
        if c in df.columns and c not in select_cols:
            select_cols.append(c)

    available = [c for c in select_cols if c in df.columns]
    flat = df.select(available)

    # ── Temporal split and export ─────────────────────────────────────────────
    export_dir = os.path.join(out_path, "exported")
    os.makedirs(export_dir, exist_ok=True)

    train_path = os.path.join(export_dir, "train_raw.parquet")
    test_path  = os.path.join(export_dir, "test_raw.parquet")

    train_df = flat.filter(F.col("origination_dt") < F.lit(TRAIN_CUTOFF))
    test_df  = flat.filter(F.col("origination_dt") >= F.lit(TRAIN_CUTOFF))

    log.info("Exporting raw train set → %s", train_path)
    train_df.write.mode("overwrite").parquet(train_path)
    log.info("Exporting raw test set → %s", test_path)
    test_df.write.mode("overwrite").parquet(test_path)

    # Save column lists for Phase 2
    meta = {
        "numeric_features": num_feats,
        "categorical_features": cat_feats,
        "binary_features": bin_feats,
        "target_col": TARGET_COL,
        "train_cutoff": TRAIN_CUTOFF,
        "seed": SEED,
    }
    meta_path = os.path.join(export_dir, "columns.json")
    with open(meta_path, "w") as fh:
        json.dump(meta, fh, indent=2)
    log.info("Saved column metadata → %s", meta_path)


# ============================================================================
#  PHASE 2:  SKLEARN PREPROCESSING + MODEL FITTING
# ============================================================================

import pyarrow.dataset as ds

def read_first_n_rows(path, n=100_000):
    dataset = ds.dataset(path, format="parquet")

    batches = []
    total = 0

    for batch in dataset.to_batches(batch_size=min(n, 1_000_000)):
        pdf = batch.to_pandas()
        batches.append(pdf)

        total += len(pdf)
        if total >= n:
            break

    return pd.concat(batches, ignore_index=True).head(n)
    
    
def load_raw_data(out_path: str, sample_frac: float = 0.1):
    """Load exported raw Parquet into pandas, optionally subsample."""
    export_dir = os.path.join(out_path, "exported")

    with open(os.path.join(export_dir, "columns.json")) as fh:
        col_meta = json.load(fh)

    log.info("Loading raw train Parquet …")
    train_pd = pd.read_parquet(os.path.join(export_dir, "train_raw.parquet"))
    #train_pd = read_first_n_rows(os.path.join(export_dir, "train_raw.parquet"), 50_000_000)
    log.info("Loading raw test Parquet …")
    test_pd = pd.read_parquet(os.path.join(export_dir, "test_raw.parquet"))
    #test_pd = read_first_n_rows(os.path.join(export_dir, "test_raw.parquet"), 5_000_000)

    log.info("Full sizes: train={:,}, test={:,}".format(len(train_pd), len(test_pd)))

    if sample_frac < 1.0:
        train_pd = train_pd.sample(frac=sample_frac, random_state=SEED)
        test_pd  = test_pd.sample(frac=sample_frac, random_state=SEED)
        log.info("After sampling (%.0f%%): train={:,}, test={:,}".format(
            len(train_pd), len(test_pd)) % (sample_frac * 100))

    return train_pd, test_pd, col_meta


def build_sklearn_preprocessor(train_pd, col_meta):
    """
    Build and fit an sklearn ColumnTransformer that handles:
      - Numeric features  → SimpleImputer(median) + StandardScaler
      - Categorical features → SimpleImputer(most_frequent) + OneHotEncoder
      - Binary features → SimpleImputer(constant=0) (pass-through after fill)

    Returns the fitted preprocessor and the feature name list.

    This preprocessor is saved as a pickle and reused identically in
    inference_pipeline_v3.py — no manual feature vector reconstruction.
    """
    num_feats = [c for c in col_meta["numeric_features"] if c in train_pd.columns]
    cat_feats = [c for c in col_meta["categorical_features"] if c in train_pd.columns]
    bin_feats = [c for c in col_meta["binary_features"] if c in train_pd.columns]

    numeric_pipe = SkPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
    ])

    categorical_pipe = SkPipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe",     SkOHE(drop="first", sparse_output=False, handle_unknown="infrequent_if_exist",
                          min_frequency=0.001)),
    ])

    binary_pipe = SkPipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipe,   num_feats),
        ("cat", categorical_pipe, cat_feats),
        ("bin", binary_pipe,    bin_feats),
    ], remainder="drop")

    log.info("Fitting sklearn preprocessor (imputer + OHE + scaler) …")
    train_pd[bin_feats] = train_pd[bin_feats].astype(np.float32)
    preprocessor.fit(train_pd)

    # ── Build feature names ───────────────────────────────────────────────────
    feature_names = list(num_feats)
    try:
        ohe = preprocessor.named_transformers_["cat"].named_steps["ohe"]
        cat_names = ohe.get_feature_names_out(cat_feats).tolist()
        feature_names += cat_names
    except Exception:
        feature_names += [f"{c}_enc" for c in cat_feats]
    feature_names += list(bin_feats)

    log.info("Preprocessor fitted: %d features total", len(feature_names))
    return preprocessor, feature_names


def prepare_arrays(train_pd, test_pd, preprocessor):
    """Transform train/test DataFrames using the fitted preprocessor."""
    log.info("Transforming train set …")
    X_train = preprocessor.transform(train_pd).astype(np.float32)
    y_train = train_pd[TARGET_COL].values.astype(np.int32)
    
    
    log.info("Transforming test set …")
    test_pd[BINARY_FEATURES] = test_pd[BINARY_FEATURES].astype(np.float32)
    X_test = preprocessor.transform(test_pd).astype(np.float32)
    y_test = test_pd[TARGET_COL].values.astype(np.int32)
    

    log.info("Feature matrix: train=%s, test=%s (%.1f GB total)",
             X_train.shape, X_test.shape,
             (X_train.nbytes + X_test.nbytes) / 1e9)
    return X_train, y_train, X_test, y_test


def compute_weight_ratio(y_train):
    """Compute positive-class up-weight = n_neg / n_pos."""
    n_neg = np.sum(y_train == 0)
    n_pos = np.sum(y_train == 1)
    ratio = n_neg / max(n_pos, 1)
    log.info("Class balance: n_neg=%d, n_pos=%d, weight_ratio=%.2f",
             n_neg, n_pos, ratio)
    return ratio


# ── MODEL A: Logistic Regression ─────────────────────────────────────────────

def train_logistic_regression(X_train, y_train, X_test, weight_ratio):
    """sklearn Logistic Regression (L2, saga solver)."""
    log.info("Training Logistic Regression …")
    t0 = time.time()
    model = LogisticRegression(
        penalty="l2",
        C=1000.0,
        solver="saga",
        max_iter=200,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
        verbose=1)
    model.fit(X_train, y_train)
    log.info("LR done in %.1f s", time.time() - t0)
    return model, model.predict_proba(X_test)[:, 1]


# ── MODEL B: Random Forest ───────────────────────────────────────────────────

def train_random_forest(X_train, y_train, X_test, weight_ratio, n_sample=None):
    """sklearn Random Forest (200 trees, depth 10)."""
    log.info("Training Random Forest …")
    t0 = time.time()

    # --- sampling ----------------------------------------------------------
    if n_sample is not None:
        X_fit, y_fit = X_train[:n_sample], y_train[:n_sample]
        log.info("Using first %d rows for training", n_sample)
    else:
        X_fit, y_fit = X_train, y_train
    # -----------------------------------------------------------------------

    model = RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=50,
        max_features="sqrt", class_weight="balanced",
        random_state=SEED, n_jobs=-1, verbose=1)
    model.fit(X_fit, y_fit)
    log.info("RF done in %.1f s", time.time() - t0)
    return model, model.predict_proba(X_test)[:, 1]


# ── MODEL C: SGD Classifier ──────────────────────────────────────────────────

def train_sgd(X_train, y_train, X_test, weight_ratio):
    """sklearn SGDClassifier with log_loss (linear model trained via SGD)."""
    log.info("Training SGD Classifier …")
    t0 = time.time()
    model = SGDClassifier(
        loss="log_loss", penalty="l2", alpha=1e-4,
        max_iter=1000, tol=1e-3, class_weight="balanced",
        random_state=SEED, n_jobs=-1, verbose=1)
    model.fit(X_train, y_train)
    log.info("SGD done in %.1f s", time.time() - t0)
    # SGDClassifier with log_loss supports predict_proba when fitted
    return model, model.predict_proba(X_test)[:, 1]


# ── MODEL D: XGBoost ─────────────────────────────────────────────────────────

def train_xgboost(X_train, y_train, X_test, y_test, weight_ratio):
    """XGBoost gradient boosting (hist method)."""
    log.info("Training XGBoost (n=%d) …", len(X_train))
    t0 = time.time()
    model = xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.01,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=weight_ratio, eval_metric="auc",
        random_state=SEED, n_jobs=-1, tree_method="hist", verbosity=0)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)
    log.info("XGBoost done in %.1f s", time.time() - t0)
    return model, model.predict_proba(X_test)[:, 1]


# ── MODEL E: LightGBM ────────────────────────────────────────────────────────

def train_lightgbm(X_train, y_train, X_test, weight_ratio):
    """LightGBM with DART booster."""
    log.info("Training LightGBM DART (n=%d) …", len(X_train))
    t0 = time.time()
    sw = np.where(y_train == 1, weight_ratio, 1.0)
    model = lgb.LGBMClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        num_leaves=31, subsample=0.8, colsample_bytree=0.8,
        min_child_samples=20, boosting_type="dart",
        random_state=SEED, n_jobs=-1)
    model.fit(X_train, y_train, sample_weight=sw,
              callbacks=[lgb.log_evaluation(period=-1)])
    log.info("LightGBM done in %.1f s", time.time() - t0)
    return model, model.predict_proba(X_test)[:, 1]


# ── MODEL F: Cox Proportional Hazards ─────────────────────────────────────────

def train_cox(abt_path: str, out_path: str, sample_frac: float = 0.1):
    """Cox PH survival model on ABT Parquet (pure pandas, no Spark)."""
    log.info("Fitting Cox PH (frac=%.2f) …", sample_frac)
    pdf = pd.read_parquet(abt_path)
    if "refi_incentive" not in pdf.columns:
        pdf["refi_incentive"] = pdf["orig_interest_rate"] - pdf["current_interest_rate"]

    cox_cols = [c for c in [
        "loan_age", "prepaid", "fico", "orig_interest_rate",
        "refi_incentive", "orig_ltv", "dti", "upb_fraction",
        "burnout", "is_refi", "term_15y", "is_investor",
        "has_ppm", "modified", "in_forbearance",
    ] if c in pdf.columns]

    pdf = pdf[cox_cols].dropna()
    pdf = pdf[(pdf["loan_age"] > 0) & pdf["prepaid"].notna()]
    if sample_frac < 1.0:
        pdf = pdf.sample(frac=sample_frac, random_state=SEED)

    log.info("Cox sample: %d rows (prepay=%.2f%%)", len(pdf), pdf["prepaid"].mean() * 100)
    cph = CoxPHFitter(penalizer=0.1)
    cph.fit(pdf, duration_col="loan_age", event_col="prepaid", show_progress=True)
    cph.print_summary()

    # Plots
    fig, ax = plt.subplots(figsize=(10, 7))
    cph.plot(ax=ax)
    ax.set_title("Cox PH – Hazard Ratios with 95% CI")
    fig.tight_layout()
    fig.savefig(os.path.join(out_path, "cox_hazard_ratios.png"), dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 5))
    cph.baseline_survival_.plot(ax=ax, color="#e74c3c", linewidth=2)
    ax.set_title("Cox PH – Baseline Survival S₀(t)")
    ax.set_xlabel("Loan Age (months)"); ax.set_ylabel("S(t)")
    fig.tight_layout()
    fig.savefig(os.path.join(out_path, "cox_baseline_survival.png"), dpi=150)
    plt.close(fig)

    return cph, pdf


# ============================================================================
#  GRID SEARCH
# ============================================================================

def grid_search_xgboost(X_train, y_train, weight_ratio, out_path,
                        sample_frac=0.05, sample_n=None):
    """
    Grid search for XGBoost on a stratified sample of (X_train, y_train).

    Parameters
    ----------
    X_train : array-like or sparse matrix
        Training features.
    y_train : array-like
        Binary target.
    weight_ratio : float
        scale_pos_weight for XGBoost.
    out_path : str
        Output path for saving grid search results.
    sample_frac : float, default=0.05
        Fraction of training data to use in grid search if sample_n is None.
    sample_n : int or None, default=None
        Absolute sample size for grid search. Overrides sample_frac if provided.

    Returns
    -------
    gs : fitted GridSearchCV object
    """

    n_total = len(y_train)
    if n_total == 0:
        raise ValueError("X_train / y_train are empty")

    y_train = np.asarray(y_train).astype(np.int32)

    # --- choose sample size ---
    if sample_n is not None:
        n_sample = min(sample_n, n_total)
    else:
        n_sample = max(1, int(n_total * sample_frac))

    # --- stratified sample ---
    if n_sample < n_total:
        sss = StratifiedShuffleSplit(
            n_splits=1,
            train_size=n_sample,
            random_state=SEED
        )
        sample_idx, _ = next(sss.split(np.zeros(n_total), y_train))
        X_gs = X_train[sample_idx]
        y_gs = y_train[sample_idx]
    else:
        X_gs = X_train
        y_gs = y_train

    # --- reduce memory for dense arrays ---
    if isinstance(X_gs, np.ndarray):
        X_gs = X_gs.astype(np.float32, copy=False)

    log.info("XGB grid sample: %s rows out of %s (%.2f%%)",
             len(y_gs), n_total, 100 * len(y_gs) / n_total)

    base = xgb.XGBClassifier(
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=weight_ratio,
        eval_metric="auc",
        random_state=SEED,
        n_jobs=-1,
        tree_method="hist",
        verbosity=0,
    )

    param_grid = {
        "n_estimators": [400, 600, 800],
        "max_depth": [4, 6],
        "learning_rate": [0.01, 0.05],
    }

    n_cfg = (
        len(param_grid["n_estimators"])
        * len(param_grid["max_depth"])
        * len(param_grid["learning_rate"])
    )
    log.info("Grid search: XGBoost (%d configs × 3 folds) …", n_cfg)

    gs = GridSearchCV(
        estimator=base,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=3,
        n_jobs=1,
        verbose=1,
        refit=True,
    )

    gs.fit(X_gs, y_gs)

    log.info("Best XGB: %s  AUC=%.4f", gs.best_params_, gs.best_score_)
    _save_sklearn_grid_results("XGBoost", gs, out_path)
    return gs
    

def grid_search_lightgbm(X_train, y_train, weight_ratio, out_path,
                         sample_frac=0.05, sample_n=None):
    """
    Grid search for LightGBM on a stratified sample of (X_train, y_train).

    Parameters
    ----------
    X_train : array-like or sparse matrix
        Training features.
    y_train : array-like
        Binary target.
    weight_ratio : float
        Positive-class weight multiplier.
    out_path : str
        Output path for saving grid search results.
    sample_frac : float, default=0.05
        Fraction of training data to use in grid search if sample_n is None.
    sample_n : int or None, default=None
        Absolute sample size for grid search. Overrides sample_frac if provided.

    Returns
    -------
    gs : fitted GridSearchCV object
    """

    n_total = len(y_train)
    if n_total == 0:
        raise ValueError("X_train / y_train are empty")

    y_train = np.asarray(y_train).astype(np.int32)

    # --- choose sample size ---
    if sample_n is not None:
        n_sample = min(sample_n, n_total)
    else:
        n_sample = max(1, int(n_total * sample_frac))

    # --- stratified sample ---
    if n_sample < n_total:
        sss = StratifiedShuffleSplit(
            n_splits=1,
            train_size=n_sample,
            random_state=SEED
        )
        sample_idx, _ = next(sss.split(np.zeros(n_total), y_train))
        X_gs = X_train[sample_idx]
        y_gs = y_train[sample_idx]
    else:
        X_gs = X_train
        y_gs = y_train

    # --- reduce memory for dense arrays ---
    if isinstance(X_gs, np.ndarray):
        X_gs = X_gs.astype(np.float32, copy=False)

    sample_weights = np.where(y_gs == 1, weight_ratio, 1.0).astype(np.float32)

    log.info("LGB grid sample: %s rows out of %s (%.2f%%)",
             len(y_gs), n_total, 100 * len(y_gs) / n_total)

    base = lgb.LGBMClassifier(
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=50,
        boosting_type="dart",
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
    )

    param_grid = {
        "n_estimators": [300, 500],
        "max_depth": [4, 6],
        "num_leaves": [31, 63],
        "learning_rate": [0.01, 0.05, 0.1],
    }

    n_cfg = (
        len(param_grid["n_estimators"])
        * len(param_grid["max_depth"])
        * len(param_grid["num_leaves"])
        * len(param_grid["learning_rate"])
    )
    log.info("Grid search: LightGBM (%d configs × 3 folds) …", n_cfg)

    gs = GridSearchCV(
        estimator=base,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=3,
        n_jobs=1,
        verbose=1,
        refit=True,
    )

    gs.fit(
        X_gs,
        y_gs,
        sample_weight=sample_weights,
        callbacks=[lgb.log_evaluation(period=0)]
    )

    log.info("Best LGB: %s  AUC=%.4f", gs.best_params_, gs.best_score_)
    _save_sklearn_grid_results("LightGBM", gs, out_path)
    return gs

    
def grid_search_lr(X_train, y_train, out_path):
    gs = GridSearchCV(
        LogisticRegression(solver="saga", max_iter=200,
            class_weight="balanced", random_state=SEED, n_jobs=-1),
        {"C": [0.01, 0.1, 1.0, 10.0, 1000.0], "penalty": ["l1", "l2"]},
        scoring="roc_auc", cv=3, n_jobs=-1, verbose=1, refit=True)
    gs.fit(X_train, y_train)
    log.info("Best LR: %s AUC=%.4f", gs.best_params_, gs.best_score_)
    _save_grid("Logistic_Regression", gs, out_path)
    return gs

def grid_search_cox(abt_path, out_path, sample_frac=0.1):
    pdf = pd.read_parquet(abt_path)
    cox_cols = [c for c in ["loan_age","prepaid","fico","orig_interest_rate",
        "refi_incentive","orig_ltv","dti","upb_fraction","burnout",
        "is_refi","term_15y","is_investor","has_ppm","modified","in_forbearance",
    ] if c in pdf.columns]
    if "refi_incentive" not in pdf.columns:
        pdf["refi_incentive"] = pdf["orig_interest_rate"] - pdf["current_interest_rate"]
    pdf = pdf[cox_cols].dropna()
    pdf = pdf[(pdf["loan_age"]>0) & pdf["prepaid"].notna()]
    if sample_frac < 1.0: pdf = pdf.sample(frac=sample_frac, random_state=SEED)
    results = []
    for pen in [0.001, 0.01, 0.1, 1.0]:
        cph = CoxPHFitter(penalizer=pen)
        cph.fit(pdf, duration_col="loan_age", event_col="prepaid")
        results.append({"penalizer": pen, "concordance_index": cph.concordance_index_})
    res_df = pd.DataFrame(results).sort_values("concordance_index", ascending=False)
    res_df.to_csv(os.path.join(out_path, "grid_search_cox.csv"), index=False)
    return res_df

# ── Grid search helpers ───────────────────────────────────────────────────

def _save_grid_results(model_name: str, grid, avg_metrics: list,
                       param_names: list, out_path: str):
    """Write PySpark CrossValidator results to CSV."""
    rows = []
    for params, metric in zip(grid, avg_metrics):
        row = {p.name: params[p] for p in params}
        row["avg_auc_roc"] = round(metric, 4)
        rows.append(row)
    df = pd.DataFrame(rows).sort_values("avg_auc_roc", ascending=False)
    fname = f"grid_search_{model_name.lower().replace(' ', '_')}.csv"
    path = os.path.join(out_path, fname)
    df.to_csv(path, index=False)
    log.info("Saved %s grid results → %s", model_name, path)


def _save_sklearn_grid_results(model_name: str, gs, out_path: str):
    """Write sklearn GridSearchCV results to CSV."""
    df = pd.DataFrame(gs.cv_results_)
    cols = [c for c in df.columns
            if c.startswith("param_") or c in ("mean_test_score", "std_test_score",
                                                "rank_test_score")]
    df = df[cols].sort_values("rank_test_score")
    fname = f"grid_search_{model_name.lower().replace(' ', '_')}.csv"
    path = os.path.join(out_path, fname)
    df.to_csv(path, index=False)
    log.info("Saved %s grid results → %s", model_name, path)



# ============================================================================
#  EVALUATION
# ============================================================================

def evaluate(y_true, y_prob, model_name: str) -> dict:
    auc   = roc_auc_score(y_true, y_prob)
    prauc = average_precision_score(y_true, y_prob)
    brier = brier_score_loss(y_true, y_prob)
    log.info("%-22s  AUC-ROC=%.4f  PR-AUC=%.4f  Brier=%.5f",
             model_name, auc, prauc, brier)
    return {"Model": model_name, "AUC-ROC": round(auc, 4),
            "PR-AUC": round(prauc, 4), "Brier": round(brier, 5)}


# ============================================================================
#  PHASE 3:  SAVING
# ============================================================================

def save_all(out_path, preprocessor=None, feature_names=None,
             lr_model=None, rf_model=None, sgd_model=None,
             xgb_model=None, lgb_model=None, cox_model=None,
             weight_ratio=1.0, col_meta=None):
    """
    Save all models and the sklearn preprocessor as pickle files.

    The preprocessor pickle contains the fitted SimpleImputer,
    OneHotEncoder, and StandardScaler — used identically at
    inference time by inference_pipeline_v3.py.
    """
    model_dir = os.path.join(out_path, "saved_models")
    os.makedirs(model_dir, exist_ok=True)

    # ── Preprocessor (the key artefact for inference) ─────────────────────────
    if preprocessor is not None:
        p = os.path.join(model_dir, "preprocessor.pkl")
        with open(p, "wb") as fh:
            pickle.dump(preprocessor, fh, protocol=pickle.HIGHEST_PROTOCOL)
        log.info("Saved sklearn preprocessor → %s", p)

    # ── Models ────────────────────────────────────────────────────────────────
    for name, model in [("lr", lr_model), ("rf", rf_model), ("sgd", sgd_model),
                        ("xgb", xgb_model), ("lgb", lgb_model), ("cox", cox_model)]:
        if model is not None:
            p = os.path.join(model_dir, f"{name}.pkl")
            with open(p, "wb") as fh:
                pickle.dump(model, fh, protocol=pickle.HIGHEST_PROTOCOL)
            log.info("Saved %s → %s", name.upper(), p)

    # ── Metadata ──────────────────────────────────────────────────────────────
    meta = {
        "feature_names": feature_names or [],
        "weight_ratio": weight_ratio,
        "numeric_features": col_meta.get("numeric_features", []) if col_meta else [],
        "categorical_features": col_meta.get("categorical_features", []) if col_meta else [],
        "binary_features": col_meta.get("binary_features", []) if col_meta else [],
        "target_col": TARGET_COL,
        "train_cutoff": TRAIN_CUTOFF,
        "seed": SEED,
    }
    with open(os.path.join(model_dir, "metadata.json"), "w") as fh:
        json.dump(meta, fh, indent=2)
    log.info("Saved metadata.json")


def save_model(model, model_dir, name):
    """Save single model to pickle."""
    if model is None:
        return

    os.makedirs(model_dir, exist_ok=True)
    path = os.path.join(model_dir, f"{name}.pkl")

    with open(path, "wb") as fh:
        pickle.dump(model, fh, protocol=pickle.HIGHEST_PROTOCOL)

    log.info("Saved %s → %s", name.upper(), path)


def load_model(path):
    with open(path, "rb") as fh:
        return pickle.load(fh)


# ============================================================================
#  VISUALISATIONS
# ============================================================================

def plot_roc_curves(results_list, out_path):
    fig, ax = plt.subplots(figsize=(9, 7))
    for name, yt, yp in results_list:
        fpr, tpr, _ = roc_curve(yt, yp)
        ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC={roc_auc_score(yt,yp):.4f})")
    ax.plot([0,1],[0,1],"k--",lw=1,label="Random")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title("ROC Curves – Prepayment Models V3")
    ax.legend(loc="lower right", fontsize=9); fig.tight_layout()
    fig.savefig(os.path.join(out_path, "roc_curves.png"), dpi=150); plt.close(fig)

def plot_pr_curves(results_list, out_path):
    fig, ax = plt.subplots(figsize=(9, 7))
    for name, yt, yp in results_list:
        prec, rec, _ = precision_recall_curve(yt, yp)
        ax.plot(rec, prec, lw=2, label=f"{name} (AP={average_precision_score(yt,yp):.4f})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("PR Curves – Prepayment Models V3")
    ax.legend(loc="upper right", fontsize=9); fig.tight_layout()
    fig.savefig(os.path.join(out_path, "pr_curves.png"), dpi=150); plt.close(fig)

def plot_calibration(results_list, out_path):
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.plot([0,1],[0,1],"k--",label="Perfect")
    for name, yt, yp in results_list:
        frac, mean = calibration_curve(yt, yp, n_bins=20, strategy="uniform")
        ax.plot(mean, frac, "s-", lw=1.5, ms=4, label=name)
    ax.set_xlabel("Mean Predicted"); ax.set_ylabel("Fraction Positive")
    ax.set_title("Calibration – V3"); ax.legend(fontsize=9); fig.tight_layout()
    fig.savefig(os.path.join(out_path, "calibration.png"), dpi=150); plt.close(fig)

def plot_feature_importance(model, feat_names, model_name, out_path, top_n=25):
    if model is None: return
    imp = model.feature_importances_
    n = min(len(imp), len(feat_names))
    fi = pd.Series(imp[:n], index=feat_names[:n]).nlargest(top_n).sort_values()
    fig, ax = plt.subplots(figsize=(9, max(5, top_n*0.35)))
    fi.plot(kind="barh", ax=ax, color="#3498db", edgecolor="black", lw=0.4)
    ax.set_title(f"{model_name} – Feature Importances (Top {top_n})")
    fig.tight_layout()
    fig.savefig(os.path.join(out_path, f"fi_{model_name.lower().replace(' ','_')}.png"), dpi=150)
    plt.close(fig)

def plot_lr_coefficients(model, feat_names, out_path, top_n=25):
    if model is None: return
    coefs = model.coef_[0]
    n = min(len(coefs), len(feat_names))
    s = pd.Series(coefs[:n], index=feat_names[:n])
    top = s.abs().nlargest(top_n).index
    sel = s[top].sort_values()
    fig, ax = plt.subplots(figsize=(9, max(5, top_n*0.35)))
    colors = ["#e74c3c" if v<0 else "#2ecc71" for v in sel]
    sel.plot(kind="barh", ax=ax, color=colors, edgecolor="black", lw=0.4)
    ax.set_title(f"LR – Top {top_n} Coefficients"); fig.tight_layout()
    fig.savefig(os.path.join(out_path, "fi_logistic_regression.png"), dpi=150)
    plt.close(fig)

def plot_kaplan_meier(test_pd, out_path):
    if "fico_bucket" not in test_pd.columns or "loan_age" not in test_pd.columns:
        return
    pdf = test_pd[["loan_age", TARGET_COL, "fico_bucket"]].dropna().copy()
    pdf = pdf[pdf["loan_age"] > 0]
    pdf.rename(columns={TARGET_COL: "prepaid"}, inplace=True)
    palette = {"SubPrime":"#e74c3c","NearPrime":"#e67e22","Prime":"#3498db","SuperPrime":"#27ae60"}
    fig, ax = plt.subplots(figsize=(11, 6))
    for bucket, color in palette.items():
        grp = pdf[(pdf["fico_bucket"]==bucket) & pdf["prepaid"].isin([0,1])]
        if len(grp) < 50: continue
        kmf = KaplanMeierFitter()
        kmf.fit(grp["loan_age"].clip(lower=1), grp["prepaid"], label=f"{bucket} (n={len(grp):,})")
        kmf.plot_survival_function(ax=ax, ci_show=True, color=color, lw=2)
    ax.set_title("KM: Survival by FICO Tier"); ax.set_xlabel("Loan Age")
    ax.set_ylabel("P(Not Prepaid)"); ax.set_ylim(0,1)
    ax.legend(fontsize=9); fig.tight_layout()
    fig.savefig(os.path.join(out_path, "kaplan_meier.png"), dpi=150); plt.close(fig)

def plot_smm_cpr(test_pd, results_list, out_path):
    if "reporting_date" not in test_pd.columns: return
    agg = test_pd.groupby("reporting_date").agg(
        n=(TARGET_COL,"count"), n_prepaid=(TARGET_COL,"sum")).reset_index()
    agg["smm"] = agg["n_prepaid"]/agg["n"]
    agg["cpr"] = (1-(1-agg["smm"])**12)*100
    agg = agg.sort_values("reporting_date")
    fig,(a1,a2) = plt.subplots(2,1,figsize=(13,10),sharex=True)
    a1.plot(agg["reporting_date"],agg["smm"]*100,color="black",lw=2.5,label="Actual SMM")
    a2.plot(agg["reporting_date"],agg["cpr"],color="black",lw=2.5,label="Actual CPR")
    colors = ["#e74c3c","#3498db","#2ecc71","#9b59b6","#e67e22","#1abc9c","#f39c12","#8e44ad"]
    for (name,yt,yp),c in zip(results_list, colors):
        m = yp.mean()
        a1.axhline(m*100,ls="--",lw=1.2,color=c,label=f"{name} ({m*100:.2f}%)")
        a2.axhline((1-(1-m)**12)*100,ls="--",lw=1.2,color=c,label=f"{name}")
    a1.set_ylabel("SMM (%)"); a1.legend(fontsize=7)
    a2.set_ylabel("CPR (%)"); a2.set_xlabel("Date"); a2.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(os.path.join(out_path,"smm_cpr.png"),dpi=150); plt.close(fig)

def plot_model_comparison(results_df, out_path):
    metrics = ["AUC-ROC","PR-AUC","Brier"]; x = np.arange(len(results_df)); w=0.25
    fig, ax = plt.subplots(figsize=(12,6))
    for i,(m,c) in enumerate(zip(metrics,["#3498db","#2ecc71","#e74c3c"])):
        v = results_df[m].values
        bars = ax.bar(x+i*w, v, w, label=m, color=c, alpha=0.85)
        for b,val in zip(bars,v):
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.002,
                    f"{val:.4f}", ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x+w); ax.set_xticklabels(results_df["Model"],rotation=20,ha="right")
    ax.set_title("Model Comparison V3"); ax.legend(); fig.tight_layout()
    fig.savefig(os.path.join(out_path,"model_comparison.png"),dpi=150); plt.close(fig)



In [2]:
panel_path = "/home/yc-user/processed_data/panel"
abt_path = "/home/yc-user/processed_data/abt"
fred_path = "/home/yc-user/hull_white/GS10.csv"
out_path = "/home/yc-user/models/"

In [3]:
#os.makedirs(out_path, exist_ok=True)

In [8]:
log.info("=" * 60)
log.info("PHASE 1: Spark feature engineering (no encoding)")
log.info("=" * 60)
spark = create_spark()
df = load_panel_spark(spark, panel_path, fred_path)
export_raw_data(df, out_path)
spark.stop()

2026-04-14 22:40:00,099 [INFO] ============================================================
2026-04-14 22:40:00,099 [INFO] PHASE 1: Spark feature engineering (no encoding)
2026-04-14 22:40:00,100 [INFO] ============================================================
26/04/14 22:40:08 WARN Utils: Your hostname, compute-vm-64-128-512-hdd-1776192960124 resolves to a loopback address: 127.0.1.1; using 10.130.0.13 instead (on interface eth0)
26/04/14 22:40:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 22:40:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-04-14 22:40:14,198 [INFO] Loading panel Parquet from /home/yc-user/processed_data/panel
2026-04-14 22:40:22,901 [INFO] Panel rows: 82,244,050                           
2026-04-14 22:40:23,501 [IN

In [7]:
sample_frac = 1
log.info("=" * 60)
log.info("PHASE 2: sklearn preprocessing + base model fitting")
log.info("=" * 60)

train_pd, test_pd, col_meta = load_raw_data(out_path, sample_frac)
preprocessor, feature_names = build_sklearn_preprocessor(train_pd, col_meta)
X_train, y_train, X_test, y_test = prepare_arrays(train_pd, test_pd, preprocessor)
weight_ratio = compute_weight_ratio(y_train)

2026-04-15 20:53:55,568 [INFO] ============================================================
2026-04-15 20:53:55,569 [INFO] PHASE 2: sklearn preprocessing + base model fitting
2026-04-15 20:53:55,570 [INFO] ============================================================
2026-04-15 20:53:55,573 [INFO] Loading raw train Parquet …
2026-04-15 20:54:09,157 [INFO] Loading raw test Parquet …
2026-04-15 20:54:10,606 [INFO] Full sizes: train=74,195,130, test=6,824,737
2026-04-15 20:54:10,637 [INFO] Fitting sklearn preprocessor (imputer + OHE + scaler) …
2026-04-15 21:08:44,411 [INFO] Preprocessor fitted: 70 features total
2026-04-15 21:08:44,412 [INFO] Transforming train set …
2026-04-15 21:10:52,680 [INFO] Transforming test set …
2026-04-15 21:11:04,771 [INFO] Feature matrix: train=(74195130, 70), test=(6824737, 70) (22.7 GB total)
2026-04-15 21:11:04,852 [INFO] Class balance: n_neg=73208593, n_pos=986537, weight_ratio=74.21


In [8]:
if preprocessor is not None:
    p = os.path.join(out_path, "preprocessor.pkl")
    with open(p, "wb") as fh:
        pickle.dump(preprocessor, fh, protocol=pickle.HIGHEST_PROTOCOL)
    log.info("Saved sklearn preprocessor → %s", p)

2026-04-15 21:11:11,633 [INFO] Saved sklearn preprocessor → /home/yc-user/models/preprocessor.pkl


In [9]:
feature_names

['fico',
 'orig_interest_rate',
 'current_interest_rate',
 'refi_incentive',
 'refi_incentive_pos',
 'rate_spread_to_10y',
 'spread_pos',
 'orig_ltv',
 'dti',
 'orig_upb',
 'upb_fraction',
 'equity_proxy',
 'loan_age',
 'age_sq',
 'burnout',
 'pct_term_elapsed',
 'orig_loan_term',
 'remaining_months_to_mat',
 'rate_duration',
 'burnout_x_refi',
 'fico_x_refi',
 'ltv_x_refi',
 'ph_delinq_count',
 'excess_principal',
 'gs10_monthly',
 'channel_C',
 'channel_R',
 'loan_purpose_P',
 'loan_purpose_R',
 'loan_purpose_infrequent_sklearn',
 'property_type_CP',
 'property_type_MH',
 'property_type_PU',
 'property_type_SF',
 'occupancy_status_P',
 'occupancy_status_S',
 'fico_bucket_Prime',
 'fico_bucket_SuperPrime',
 'fico_bucket_infrequent_sklearn',
 'seasoning_bucket_120m+',
 'seasoning_bucket_13-36m',
 'seasoning_bucket_37-60m',
 'seasoning_bucket_61-120m',
 'month_of_year_10',
 'month_of_year_11',
 'month_of_year_12',
 'month_of_year_2',
 'month_of_year_3',
 'month_of_year_4',
 'month_of_ye

If already have `preprocessor`

In [3]:
preprocessor = load_model(os.path.join(out_path, "preprocessor.pkl"))
feature_names = ['fico',
 'orig_interest_rate',
 'current_interest_rate',
 'refi_incentive',
 'refi_incentive_pos',
 'rate_spread_to_10y',
 'spread_pos',
 'orig_ltv',
 'dti',
 'orig_upb',
 'upb_fraction',
 'equity_proxy',
 'loan_age',
 'age_sq',
 'burnout',
 'pct_term_elapsed',
 'orig_loan_term',
 'remaining_months_to_mat',
 'rate_duration',
 'burnout_x_refi',
 'fico_x_refi',
 'ltv_x_refi',
 'ph_delinq_count',
 'excess_principal',
 'gs10_monthly',
 'channel_C',
 'channel_R',
 'loan_purpose_P',
 'loan_purpose_R',
 'loan_purpose_infrequent_sklearn',
 'property_type_CP',
 'property_type_MH',
 'property_type_PU',
 'property_type_SF',
 'occupancy_status_P',
 'occupancy_status_S',
 'fico_bucket_Prime',
 'fico_bucket_SuperPrime',
 'fico_bucket_infrequent_sklearn',
 'seasoning_bucket_120m+',
 'seasoning_bucket_13-36m',
 'seasoning_bucket_37-60m',
 'seasoning_bucket_61-120m',
 'month_of_year_10',
 'month_of_year_11',
 'month_of_year_12',
 'month_of_year_2',
 'month_of_year_3',
 'month_of_year_4',
 'month_of_year_5',
 'month_of_year_6',
 'month_of_year_7',
 'month_of_year_8',
 'month_of_year_9',
 'vintage_year_2014',
 'vintage_year_infrequent_sklearn',
 'high_ltv',
 'term_15y',
 'is_refi',
 'is_cashout',
 'is_io',
 'has_ppm',
 'modified',
 'is_investor',
 'is_high_bal',
 'first_time_buyer',
 'in_forbearance',
 'has_deferral',
 'is_judicial_state',
 'is_hltv_refi']
sample_frac = 1
log.info("=" * 60)
log.info("PHASE 2: sklearn preprocessing + base model fitting")
log.info("=" * 60)
train_pd, test_pd, col_meta = load_raw_data(out_path, sample_frac)
X_train, y_train, X_test, y_test = prepare_arrays(train_pd, test_pd, preprocessor)
weight_ratio = compute_weight_ratio(y_train)

2026-04-17 14:29:56,028 [INFO] ============================================================
2026-04-17 14:29:56,029 [INFO] PHASE 2: sklearn preprocessing + base model fitting
2026-04-17 14:29:56,029 [INFO] ============================================================
2026-04-17 14:29:56,030 [INFO] Loading raw train Parquet …
2026-04-17 14:30:03,219 [INFO] Loading raw test Parquet …
2026-04-17 14:30:04,517 [INFO] Full sizes: train=74,195,130, test=6,824,737
2026-04-17 14:30:04,518 [INFO] Transforming train set …
2026-04-17 14:32:13,327 [INFO] Transforming test set …
2026-04-17 14:32:25,447 [INFO] Feature matrix: train=(74195130, 70), test=(6824737, 70) (22.7 GB total)
2026-04-17 14:32:25,531 [INFO] Class balance: n_neg=73208593, n_pos=986537, weight_ratio=74.21


### Model fit

#### Logistic regression

In [4]:
log.info("Logistic Regression")
lr_model, lr_p = train_logistic_regression(X_train, y_train, X_test, weight_ratio)
m_lr = evaluate(y_test, lr_p, "Logistic Regression")
save_model(lr_model, model_dir, "lr")

2026-04-16 21:29:27,375 [INFO] Logistic Regression
2026-04-16 21:29:27,376 [INFO] Training Logistic Regression …
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
2026-04-16 21:57:54,848 [INFO] LR done in 1707.5 s


convergence after 18 epochs took 1684 seconds


2026-04-16 21:57:58,322 [INFO] Logistic Regression     AUC-ROC=0.6940  PR-AUC=0.0314  Brier=0.21194


NameError: name 'model_dir' is not defined

In [5]:
model_dir = os.path.join(out_path, "saved_models")
save_model(lr_model, model_dir, "lr")

2026-04-16 22:04:04,693 [INFO] Saved LR → /home/yc-user/models/saved_models/lr.pkl


#### Random Forest

In [11]:
log.info("Random Forest")
rf_model, rf_p = train_random_forest(X_train, y_train, X_test, weight_ratio, n_sample=10_000_000)
m_rf = evaluate(y_test, rf_p, "Random Forest")
model_dir = os.path.join(out_path, "saved_models")
save_model(rf_model, model_dir, "rf")

2026-04-16 22:07:01,417 [INFO] Random Forest
2026-04-16 22:07:01,418 [INFO] Training Random Forest …
2026-04-16 22:07:01,419 [INFO] Using first 10000000 rows for training
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 64 concurrent workers.
[Parallel(n_jobs=-1)]: Done  72 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:  3.0min finished
2026-04-16 22:10:03,646 [INFO] RF done in 182.2 s
[Parallel(n_jobs=64)]: Using backend ThreadingBackend with 64 concurrent workers.
[Parallel(n_jobs=64)]: Done  72 tasks      | elapsed:    2.0s
[Parallel(n_jobs=64)]: Done 300 out of 300 | elapsed:    5.7s finished
2026-04-16 22:10:12,971 [INFO] Random Forest           AUC-ROC=0.7117  PR-AUC=0.0496  Brier=0.20538
2026-04-16 22:10:13,042 [INFO] Saved RF → /home/yc-user/models/saved_models/rf.pkl


#### SGD

In [12]:
sgd_model, sgd_p = train_sgd(X_train, y_train, X_test, weight_ratio)
m_sgd = evaluate(y_test, sgd_p, "SGD Classifier")
model_dir = os.path.join(out_path, "saved_models")
save_model(sgd_model, model_dir, "sgd")

2026-04-16 22:10:37,215 [INFO] Training SGD Classifier …


-- Epoch 1
Norm: 3.66, NNZs: 67, Bias: 1.914803, T: 74195130, Avg. loss: 0.822640, Objective: 0.839145
Total training time: 37.98 seconds.
-- Epoch 2
Norm: 3.69, NNZs: 67, Bias: 1.898954, T: 148390260, Avg. loss: 0.644441, Objective: 0.645114
Total training time: 75.88 seconds.
-- Epoch 3
Norm: 3.66, NNZs: 67, Bias: 1.896560, T: 222585390, Avg. loss: 0.642670, Objective: 0.643344
Total training time: 113.82 seconds.
-- Epoch 4
Norm: 3.66, NNZs: 67, Bias: 1.891557, T: 296780520, Avg. loss: 0.642005, Objective: 0.642676
Total training time: 151.78 seconds.
-- Epoch 5
Norm: 3.66, NNZs: 67, Bias: 1.905529, T: 370975650, Avg. loss: 0.641667, Objective: 0.642337
Total training time: 189.84 seconds.
-- Epoch 6
Norm: 3.67, NNZs: 67, Bias: 1.899852, T: 445170780, Avg. loss: 0.641460, Objective: 0.642129
Total training time: 227.92 seconds.
-- Epoch 7
Norm: 3.66, NNZs: 67, Bias: 1.914835, T: 519365910, Avg. loss: 0.641310, Objective: 0.641982
Total training time: 265.97 seconds.
-- Epoch 8


2026-04-16 22:16:02,381 [INFO] SGD done in 325.2 s


Norm: 3.66, NNZs: 67, Bias: 1.909103, T: 593561040, Avg. loss: 0.641236, Objective: 0.641907
Total training time: 304.07 seconds.
Convergence after 8 epochs took 304.07 seconds


2026-04-16 22:16:06,022 [INFO] SGD Classifier          AUC-ROC=0.6920  PR-AUC=0.0307  Brier=0.21161
2026-04-16 22:16:06,023 [INFO] Saved SGD → /home/yc-user/models/saved_models/sgd.pkl


#### XG Boosting

In [10]:
log.info("XGBoost")
xgb_model, xgb_p = train_xgboost(X_train, y_train, X_test, y_test, weight_ratio)
m_xgb = evaluate(y_test, xgb_p, "XGBoost")
save_model(xgb_model, model_dir, "xgb")

2026-04-15 21:25:55,207 [INFO] XGBoost
2026-04-15 21:25:55,208 [INFO] Training XGBoost (n=74195130) …


[0]	validation_0-auc:0.66779
[50]	validation_0-auc:0.68819
[100]	validation_0-auc:0.69323
[150]	validation_0-auc:0.69846
[200]	validation_0-auc:0.70247
[250]	validation_0-auc:0.70533
[300]	validation_0-auc:0.70752
[350]	validation_0-auc:0.70921
[399]	validation_0-auc:0.71053


2026-04-15 21:31:44,693 [INFO] XGBoost done in 349.5 s
2026-04-15 21:31:48,277 [INFO] XGBoost                 AUC-ROC=0.7105  PR-AUC=0.0417  Brier=0.20963
2026-04-15 21:31:48,284 [INFO] Saved XGB → /home/yc-user/models/saved_models/xgb.pkl


#### Light GBM

In [11]:
log.info("LightGBM")
lgb_model, lgb_p = train_lightgbm(X_train, y_train, X_test, weight_ratio)
m_lgb = evaluate(y_test, lgb_p, "LightGBM")
save_model(lgb_model, model_dir, "lgb")

2026-04-15 21:32:13,796 [INFO] LightGBM
2026-04-15 21:32:13,797 [INFO] Training LightGBM DART (n=74195130) …


[LightGBM] [Info] Number of positive: 986537, number of negative: 73208593
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.919381 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3609
[LightGBM] [Info] Number of data points in the train set: 74195130, number of used features: 67
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

2026-04-15 21:36:39,338 [INFO] LightGBM done in 265.5 s
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026-04-15 21:36:43,558 [INFO] LightGBM                AUC-ROC=0.7088  PR-AUC=0.0413  Brier=0.21124
2026-04-15 21:36:43,563 [INFO] Saved LGB → /home/yc-user/models/saved_models/lgb.pkl


### Grid Search

#### XG Boost

In [12]:
grid_search_xgboost(X_train, y_train, weight_ratio, out_path)

2026-04-15 22:22:08,164 [INFO] XGB grid sample: 3709756 rows out of 74195130 (5.00%)
2026-04-15 22:22:08,165 [INFO] Grid search: XGBoost (12 configs × 3 folds) …


Fitting 3 folds for each of 12 candidates, totalling 36 fits


2026-04-15 22:30:41,507 [INFO] Best XGB: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 600}  AUC=0.7098
2026-04-15 22:30:41,513 [INFO] Saved XGBoost grid results → /home/yc-user/models/grid_search_xgboost.csv


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.01, 0.05], 'max_depth': [4, 6], 'n_estimators': [400, 600, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : t

In [27]:
def grid_search_xgboost_random(X_train, y_train, weight_ratio, out_path,
                        sample_frac=0.1, sample_n=None, n_iter=25):
    """
    Randomized search for XGBoost on a stratified sample of (X_train, y_train).
    """
    from sklearn.model_selection import RandomizedSearchCV, StratifiedShuffleSplit, StratifiedKFold
    import numpy as np

    n_total = len(y_train)
    if n_total == 0:
        raise ValueError("X_train / y_train are empty")

    y_train = np.asarray(y_train).astype(np.int32)

    if sample_n is not None:
        n_sample = min(sample_n, n_total)
    else:
        n_sample = max(1, int(n_total * sample_frac))

    if n_sample < n_total:
        sss = StratifiedShuffleSplit(
            n_splits=1,
            train_size=n_sample,
            random_state=SEED
        )
        sample_idx, _ = next(sss.split(np.zeros(n_total), y_train))
        X_gs = X_train[sample_idx]
        y_gs = y_train[sample_idx]
    else:
        X_gs = X_train
        y_gs = y_train

    if isinstance(X_gs, np.ndarray):
        X_gs = X_gs.astype(np.float32, copy=False)

    log.info("XGB search sample: %s rows out of %s (%.2f%%)",
             len(y_gs), n_total, 100 * len(y_gs) / n_total)
    log.info("Positive rate in sample: %.4f", y_gs.mean())

    base = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=weight_ratio,
        tree_method="hist",
        random_state=SEED,
        n_jobs=-1,
        verbosity=0,
    )

    param_dist = {
        "n_estimators": [200, 400, 600, 800],
        "learning_rate": [0.03, 0.05, 0.1],
        "max_depth": [2, 3, 4, 5, 6],
        "min_child_weight": [1, 3, 5, 10],
        "subsample": [0.6, 0.8, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "gamma": [0, 0.1, 0.3, 1.0],
        "reg_alpha": [0, 0.01, 0.1, 1.0],
        "reg_lambda": [1, 3, 5, 10],
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

    search = RandomizedSearchCV(
        estimator=base,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring="roc_auc",
        cv=cv,
        verbose=1,
        random_state=SEED,
        n_jobs=-1,
        refit=True,
    )

    search.fit(X_gs, y_gs)

    log.info("Best XGB: %s  AUC=%.4f", search.best_params_, search.best_score_)
    _save_sklearn_grid_results("XGBoost_random", search, out_path)
    return search
    
grid_search_xgboost_random(X_train, y_train, weight_ratio, out_path)

2026-04-17 11:02:30,399 [INFO] XGB search sample: 3709756 rows out of 37097565 (10.00%)
2026-04-17 11:02:30,402 [INFO] Positive rate in sample: 0.0133


Fitting 3 folds for each of 25 candidates, totalling 75 fits


2026-04-17 11:09:21,624 [INFO] Best XGB: {'subsample': 0.8, 'reg_lambda': 10, 'reg_alpha': 0.01, 'n_estimators': 800, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 0.8}  AUC=0.7123
2026-04-17 11:09:21,630 [INFO] Saved XGBoost_random grid results → /home/yc-user/models/grid_search_xgboost_random.csv


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.6, 0.8, ...], 'gamma': [0, 0.1, ...], 'learning_rate': [0.03, 0.05, ...], 'max_depth': [2, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",25
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here..

In [29]:
def grid_search_xgboost_random_ver2(X_train, y_train, weight_ratio, out_path,
                        sample_frac=0.1, sample_n=None, n_iter=40):
    """
    Randomized search for XGBoost on a stratified sample of (X_train, y_train).
    """
    from sklearn.model_selection import RandomizedSearchCV, StratifiedShuffleSplit, StratifiedKFold
    import numpy as np

    n_total = len(y_train)
    if n_total == 0:
        raise ValueError("X_train / y_train are empty")

    y_train = np.asarray(y_train).astype(np.int32)

    if sample_n is not None:
        n_sample = min(sample_n, n_total)
    else:
        n_sample = max(1, int(n_total * sample_frac))

    if n_sample < n_total:
        sss = StratifiedShuffleSplit(
            n_splits=1,
            train_size=n_sample,
            random_state=SEED
        )
        sample_idx, _ = next(sss.split(np.zeros(n_total), y_train))
        X_gs = X_train[sample_idx]
        y_gs = y_train[sample_idx]
    else:
        X_gs = X_train
        y_gs = y_train

    if isinstance(X_gs, np.ndarray):
        X_gs = X_gs.astype(np.float32, copy=False)

    log.info("XGB search sample: %s rows out of %s (%.2f%%)",
             len(y_gs), n_total, 100 * len(y_gs) / n_total)
    log.info("Positive rate in sample: %.4f", y_gs.mean())

    base = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=weight_ratio,
        tree_method="hist",
        random_state=SEED,
        n_jobs=-1,
        verbosity=0,
    )

    param_dist = {
        "n_estimators": [700, 900, 1100, 1400, 1800],
        "learning_rate": [0.03, 0.05, 0.07, 0.1],
        "max_depth": [2, 3, 4],
        "min_child_weight": [2, 3, 4, 5, 6],
        "subsample": [0.7, 0.8, 0.9],
        "colsample_bytree": [0.7, 0.8, 0.9],
        "reg_alpha": [0.001, 0.01, 0.05, 0.1],
        "reg_lambda": [7, 10, 15, 20],
        "gamma": [0, 0.05, 0.1, 0.2],
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

    search = RandomizedSearchCV(
        estimator=base,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring="roc_auc",
        cv=cv,
        verbose=1,
        random_state=SEED,
        n_jobs=-1,
        refit=True,
    )

    search.fit(X_gs, y_gs)

    log.info("Best XGB: %s  AUC=%.4f", search.best_params_, search.best_score_)
    _save_sklearn_grid_results("XGBoost_random_2", search, out_path)
    return search
    
grid_search_xgboost_random_ver2(X_train, y_train, weight_ratio, out_path)

2026-04-17 11:24:58,173 [INFO] XGB search sample: 3709756 rows out of 37097565 (10.00%)
2026-04-17 11:24:58,176 [INFO] Positive rate in sample: 0.0133


Fitting 3 folds for each of 40 candidates, totalling 120 fits


2026-04-17 11:43:51,819 [INFO] Best XGB: {'subsample': 0.7, 'reg_lambda': 10, 'reg_alpha': 0.01, 'n_estimators': 1100, 'min_child_weight': 3, 'max_depth': 4, 'learning_rate': 0.03, 'gamma': 0.1, 'colsample_bytree': 0.7}  AUC=0.7132
2026-04-17 11:43:51,822 [INFO] Saved XGBoost_random_2 grid results → /home/yc-user/models/grid_search_xgboost_random_2.csv


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.7, 0.8, ...], 'gamma': [0, 0.05, ...], 'learning_rate': [0.03, 0.05, ...], 'max_depth': [2, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",40
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.

#### Light GBM

In [15]:
grid_search_lightgbm(X_train, y_train, weight_ratio, out_path)

2026-04-15 22:32:02,908 [INFO] LGB grid sample: 3709756 rows out of 74195130 (5.00%)
2026-04-15 22:32:02,909 [INFO] Grid search: LightGBM (24 configs × 3 folds) …


Fitting 3 folds for each of 24 candidates, totalling 72 fits


/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with f

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","LGBMClassifie...8, verbose=-1)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.01, 0.05, ...], 'max_depth': [4, 6], 'n_estimators': [300, 500], 'num_leaves': [31, 63]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candida

#### XG Boost

In [13]:
def train_xgboost(X_train, y_train, X_test, y_test, weight_ratio):
    """XGBoost gradient boosting (hist method)."""
    log.info("Training XGBoost (n=%d) …", len(X_train))
    t0 = time.time()
    model = xgb.XGBClassifier(
        n_estimators=600, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=weight_ratio, eval_metric="auc",
        random_state=SEED, n_jobs=-1, tree_method="hist", verbosity=0)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)
    log.info("XGBoost done in %.1f s", time.time() - t0)
    return model, model.predict_proba(X_test)[:, 1]

log.info("XGBoost")
xgb_model, xgb_p = train_xgboost(X_train, y_train, X_test, y_test, weight_ratio)
m_xgb = evaluate(y_test, xgb_p, "XGBoost")

2026-04-16 22:16:39,038 [INFO] XGBoost
2026-04-16 22:16:39,039 [INFO] Training XGBoost (n=74195130) …


[0]	validation_0-auc:0.66779
[50]	validation_0-auc:0.70617
[100]	validation_0-auc:0.71318
[150]	validation_0-auc:0.71571
[200]	validation_0-auc:0.71726
[250]	validation_0-auc:0.71841
[300]	validation_0-auc:0.71930
[350]	validation_0-auc:0.72005
[400]	validation_0-auc:0.72067
[450]	validation_0-auc:0.72127
[500]	validation_0-auc:0.72165
[550]	validation_0-auc:0.72209
[599]	validation_0-auc:0.72247


2026-04-16 22:26:05,890 [INFO] XGBoost done in 566.9 s
2026-04-16 22:26:10,151 [INFO] XGBoost                 AUC-ROC=0.7225  PR-AUC=0.0446  Brier=0.19972


In [14]:
model_dir = os.path.join(out_path, "saved_models")
save_model(xgb_model, model_dir, "xgb")

2026-04-16 22:26:10,165 [INFO] Saved XGB → /home/yc-user/models/saved_models/xgb.pkl


In [33]:
def train_xgboost(X_train, y_train, X_test, y_test, weight_ratio):
    """XGBoost gradient boosting (hist method)."""
    log.info("Training XGBoost from random Strat Sample (n=%d) …", len(X_train))
    t0 = time.time()
    model = xgb.XGBClassifier(
        n_estimators=800,
        reg_lambda=10,
        reg_alpha=0.01,
        max_depth=3,
        gamma=0,
        min_child_weight=3,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=weight_ratio,
        eval_metric="auc",
        random_state=SEED,
        n_jobs=-1,
        tree_method="hist",
        verbosity=0)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)
    log.info("XGBoost done in %.1f s", time.time() - t0)
    return model, model.predict_proba(X_test)[:, 1]

log.info("XGBoost")
xgb_model, xgb_p = train_xgboost(X_train, y_train, X_test, y_test, weight_ratio)
m_xgb = evaluate(y_test, xgb_p, "XGBoost")

2026-04-17 12:19:58,690 [INFO] XGBoost
2026-04-17 12:19:58,691 [INFO] Training XGBoost from random Strat Sample (n=74195130) …


[0]	validation_0-auc:0.64522
[50]	validation_0-auc:0.70842
[100]	validation_0-auc:0.71358
[150]	validation_0-auc:0.71593
[200]	validation_0-auc:0.71747
[250]	validation_0-auc:0.71861
[300]	validation_0-auc:0.71953
[350]	validation_0-auc:0.72035
[400]	validation_0-auc:0.72082
[450]	validation_0-auc:0.72121
[500]	validation_0-auc:0.72160
[550]	validation_0-auc:0.72180
[600]	validation_0-auc:0.72220
[650]	validation_0-auc:0.72241
[700]	validation_0-auc:0.72261
[750]	validation_0-auc:0.72281
[799]	validation_0-auc:0.72301


2026-04-17 12:32:30,478 [INFO] XGBoost done in 751.8 s
2026-04-17 12:32:34,741 [INFO] XGBoost                 AUC-ROC=0.7230  PR-AUC=0.0447  Brier=0.19777


In [34]:

def train_xgboost(X_train, y_train, X_test, y_test, weight_ratio):
    """XGBoost gradient boosting (hist method)."""
    log.info("Training XGBoost from random Strat Sample (n=%d) …", len(X_train))
    t0 = time.time()
    model = xgb.XGBClassifier(
        n_estimators=1100,
        reg_lambda=10,
        reg_alpha=0.01,
        max_depth=4,
        gamma=0.1,
        min_child_weight=3,
        learning_rate=0.1,
        subsample=0.7,
        colsample_bytree=0.7,
        scale_pos_weight=weight_ratio,
        eval_metric="auc",
        random_state=SEED,
        n_jobs=-1,
        tree_method="hist",
        verbosity=0)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)
    log.info("XGBoost done in %.1f s", time.time() - t0)
    return model, model.predict_proba(X_test)[:, 1]

log.info("XGBoost Grid Search")
xgb_model, xgb_p = train_xgboost(X_train, y_train, X_test, y_test, weight_ratio)
m_xgb = evaluate(y_test, xgb_p, "XGBoost")

2026-04-17 12:32:34,746 [INFO] XGBoost Grid Search
2026-04-17 12:32:34,747 [INFO] Training XGBoost from random Strat Sample (n=74195130) …


[0]	validation_0-auc:0.66779
[50]	validation_0-auc:0.71254
[100]	validation_0-auc:0.71697
[150]	validation_0-auc:0.71901
[200]	validation_0-auc:0.72043
[250]	validation_0-auc:0.72147
[300]	validation_0-auc:0.72212
[350]	validation_0-auc:0.72271
[400]	validation_0-auc:0.72323
[450]	validation_0-auc:0.72360
[500]	validation_0-auc:0.72390
[550]	validation_0-auc:0.72418
[600]	validation_0-auc:0.72439
[650]	validation_0-auc:0.72447
[700]	validation_0-auc:0.72464
[750]	validation_0-auc:0.72478
[800]	validation_0-auc:0.72495
[850]	validation_0-auc:0.72509
[900]	validation_0-auc:0.72511
[950]	validation_0-auc:0.72519
[1000]	validation_0-auc:0.72529
[1050]	validation_0-auc:0.72539
[1099]	validation_0-auc:0.72548


2026-04-17 12:51:12,097 [INFO] XGBoost done in 1117.3 s
2026-04-17 12:51:17,084 [INFO] XGBoost                 AUC-ROC=0.7255  PR-AUC=0.0459  Brier=0.19678


In [35]:
model_dir = os.path.join(out_path, "saved_models")
save_model(xgb_model, model_dir, "xgb")

2026-04-17 12:52:22,198 [INFO] Saved XGB → /home/yc-user/models/saved_models/xgb.pkl


#### Light GBM

In [15]:
def train_lightgbm(X_train, y_train, X_test, weight_ratio):
    """LightGBM with DART booster."""
    log.info("Training LightGBM DART (n=%d) …", len(X_train))
    t0 = time.time()
    sw = np.where(y_train == 1, weight_ratio, 1.0)
    model = lgb.LGBMClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.1,
        num_leaves=31, subsample=0.8, colsample_bytree=0.8,
        min_child_samples=20, boosting_type="dart",
        random_state=SEED, n_jobs=-1)
    model.fit(X_train, y_train, sample_weight=sw,
              callbacks=[lgb.log_evaluation(period=-1)])
    log.info("LightGBM done in %.1f s", time.time() - t0)
    return model, model.predict_proba(X_test)[:, 1]

log.info("LightGBM")
lgb_model, lgb_p = train_lightgbm(X_train, y_train, X_test, weight_ratio)
m_lgb = evaluate(y_test, lgb_p, "LightGBM")

2026-04-16 22:26:10,172 [INFO] LightGBM
2026-04-16 22:26:10,172 [INFO] Training LightGBM DART (n=74195130) …


[LightGBM] [Info] Number of positive: 986537, number of negative: 73208593
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.964492 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3609
[LightGBM] [Info] Number of data points in the train set: 74195130, number of used features: 67
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


2026-04-16 22:41:22,524 [INFO] LightGBM done in 912.4 s
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026-04-16 22:41:29,651 [INFO] LightGBM                AUC-ROC=0.7221  PR-AUC=0.0450  Brier=0.20204


In [17]:
save_model(lgb_model, model_dir, "lgb")

2026-04-16 22:41:39,206 [INFO] Saved LGB → /home/yc-user/models/saved_models/lgb.pkl


### Optuna

#### Light GBM

In [10]:
def grid_search_lightgbm_random(X_train, y_train, weight_ratio, out_path,
                                 sample_frac=0.05, sample_n=None, n_iter=40):
    import optuna
    from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
    from sklearn.metrics import roc_auc_score
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    n_total = len(y_train)
    if n_total == 0:
        raise ValueError("X_train / y_train are empty")
    y_train = np.asarray(y_train).astype(np.int32)

    # --- sample ---
    if sample_n is not None:
        n_sample = min(sample_n, n_total)
    else:
        n_sample = max(1, int(n_total * sample_frac))

    if n_sample < n_total:
        sss = StratifiedShuffleSplit(n_splits=1, train_size=n_sample, random_state=SEED)
        idx, _ = next(sss.split(np.zeros(n_total), y_train))
        X_gs, y_gs = X_train[idx], y_train[idx]
    else:
        X_gs, y_gs = X_train, y_train

    if isinstance(X_gs, np.ndarray):
        X_gs = X_gs.astype(np.float32, copy=False)

    sample_weights = np.where(y_gs == 1, weight_ratio, 1.0).astype(np.float32)
    log.info("LGB search sample: %d / %d (%.2f%%) | pos rate: %.4f",
             len(y_gs), n_total, 100 * len(y_gs) / n_total, y_gs.mean())

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

    def objective(trial):
        params = dict(
            objective        = "binary",
            metric           = "auc",
            boosting_type    = "gbdt",          # dart убран — главный тормоз
            random_state     = SEED,
            n_jobs           = 1,               # n_jobs=-1 только в study.optimize
            verbose          = -1,
            n_estimators     = trial.suggest_int("n_estimators", 100, 1000),            
            learning_rate    = trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
            max_depth        = trial.suggest_int("max_depth", 3, 7),
            num_leaves       = trial.suggest_int("num_leaves", 15, 127),
            min_child_samples= trial.suggest_int("min_child_samples", 20, 200),
            subsample        = trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0),
            reg_alpha        = trial.suggest_float("reg_alpha", 1e-3, 1.0, log=True),
            reg_lambda       = trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
            min_split_gain   = trial.suggest_float("min_split_gain", 0.0, 0.1),
        )

        scores = []
        for train_idx, val_idx in cv.split(X_gs, y_gs):
            X_tr, X_val = X_gs[train_idx], X_gs[val_idx]
            y_tr, y_val = y_gs[train_idx], y_gs[val_idx]
            sw_tr = sample_weights[train_idx]

            model = lgb.LGBMClassifier(**params)
            model.fit(
                X_tr, y_tr,
                sample_weight=sw_tr,
                eval_set=[(X_val, y_val)],
                callbacks=[
                    lgb.early_stopping(30, verbose=False),
                    lgb.log_evaluation(period=0),
                ],
            )
            scores.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))

        return np.mean(scores)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study.optimize(objective, n_trials=n_iter, n_jobs=-1, show_progress_bar=True)

    log.info("Best LGB: %s  AUC=%.4f", study.best_params, study.best_value)
    return study

grid_search_lightgbm_random(X_train, y_train, weight_ratio, out_path)

2026-04-17 19:28:39,682 [INFO] LGB search sample: 3709756 / 74195130 (5.00%) | pos rate: 0.0133
  0%|          | 0/40 [00:00<?, ?it/s]/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/yc-user/venvs/thesis/lib/python3.12/

#### SGD

In [11]:
def grid_search_sgd(X_train, y_train, weight_ratio, out_path,
                    sample_frac=0.1, sample_n=None, n_iter=30):
    import optuna
    from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
    from sklearn.metrics import roc_auc_score
    from sklearn.linear_model import SGDClassifier
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    n_total = len(y_train)
    if n_total == 0:
        raise ValueError("X_train / y_train are empty")
    y_train = np.asarray(y_train).astype(np.int32)

    # --- sample ---
    if sample_n is not None:
        n_sample = min(sample_n, n_total)
    else:
        n_sample = max(1, int(n_total * sample_frac))

    if n_sample < n_total:
        sss = StratifiedShuffleSplit(n_splits=1, train_size=n_sample, random_state=SEED)
        idx, _ = next(sss.split(np.zeros(n_total), y_train))
        X_gs, y_gs = X_train[idx], y_train[idx]
    else:
        X_gs, y_gs = X_train, y_train

    if isinstance(X_gs, np.ndarray):
        X_gs = X_gs.astype(np.float32, copy=False)

    sample_weights = np.where(y_gs == 1, weight_ratio, 1.0).astype(np.float32)
    log.info("SGD search sample: %d / %d (%.2f%%) | pos rate: %.4f",
             len(y_gs), n_total, 100 * len(y_gs) / n_total, y_gs.mean())

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

    def objective(trial):
        penalty = trial.suggest_categorical("penalty", ["l1", "l2", "elasticnet"])

        params = dict(
            loss         = trial.suggest_categorical("loss", ["log_loss", "modified_huber"]),
            penalty      = penalty,
            alpha        = trial.suggest_float("alpha", 1e-6, 1e-1, log=True),
            max_iter     = trial.suggest_int("max_iter", 500, 2000, step=500),
            tol          = trial.suggest_float("tol", 1e-4, 1e-2, log=True),
            learning_rate= trial.suggest_categorical("learning_rate", ["optimal", "adaptive", "invscaling"]),
            class_weight = "balanced",
            random_state = SEED,
            n_jobs       = -1,
        )

        # l1_ratio только для elasticnet
        if penalty == "elasticnet":
            params["l1_ratio"] = trial.suggest_float("l1_ratio", 0.0, 1.0)

        # eta0 нужен только для adaptive/invscaling
        if params["learning_rate"] in ("adaptive", "invscaling"):
            params["eta0"] = trial.suggest_float("eta0", 1e-4, 1e-1, log=True)

        # SGD чувствителен к масштабу — скейлим внутри CV чтобы не было утечки
        pipeline = Pipeline([
            ("scaler", StandardScaler(with_mean=False)),  # with_mean=False для sparse
            ("clf", SGDClassifier(**params)),
        ])

        scores = []
        for train_idx, val_idx in cv.split(X_gs, y_gs):
            X_tr, X_val = X_gs[train_idx], X_gs[val_idx]
            y_tr, y_val = y_gs[train_idx], y_gs[val_idx]
            sw_tr = sample_weights[train_idx]

            pipeline.fit(X_tr, y_tr, clf__sample_weight=sw_tr)
            scores.append(roc_auc_score(y_val, pipeline.predict_proba(X_val)[:, 1]))

        return np.mean(scores)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(objective, n_trials=n_iter, n_jobs=-1, show_progress_bar=True)

    log.info("Best SGD: %s  AUC=%.4f", study.best_params, study.best_value)
    return study
grid_search_sgd(X_train, y_train, weight_ratio, out_path)

2026-04-17 19:45:28,307 [INFO] SGD search sample: 7419513 / 74195130 (10.00%) | pos rate: 0.0133
Best trial: 11. Best value: 0.679737: 100%|██████████| 30/30 [31:40<00:00, 63.36s/it] 
2026-04-17 20:17:09,235 [INFO] Best SGD: {'penalty': 'l1', 'loss': 'log_loss', 'alpha': 0.001104792875602074, 'max_iter': 1000, 'tol': 0.0024383253286981705, 'learning_rate': 'adaptive', 'eta0': 0.00026373750404705816}  AUC=0.6797


In [12]:
def train_sgd(X_train, y_train, X_test, weight_ratio):
    """sklearn SGDClassifier with log_loss (linear model trained via SGD)."""
    log.info("Training SGD Classifier …")
    t0 = time.time()
    model = SGDClassifier(
        loss="log_loss",
        penalty="l1",
        alpha=1e-3,
        max_iter=1000,
        tol=2e-3,
        eta0=2e-4,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
        verbose=1)
    model.fit(X_train, y_train)
    log.info("SGD done in %.1f s", time.time() - t0)
    # SGDClassifier with log_loss supports predict_proba when fitted
    return model, model.predict_proba(X_test)[:, 1]
sgd_model, sgd_p = train_sgd(X_train, y_train, X_test, weight_ratio)
m_sgd = evaluate(y_test, sgd_p, "SGD Classifier")

2026-04-17 20:18:38,784 [INFO] Training SGD Classifier …


-- Epoch 1
Norm: 3.24, NNZs: 24, Bias: -0.058904, T: 74195130, Avg. loss: 0.682965, Objective: 0.697429
Total training time: 56.09 seconds.
-- Epoch 2
Norm: 1.80, NNZs: 25, Bias: -0.064802, T: 148390260, Avg. loss: 0.645850, Objective: 0.652350
Total training time: 111.63 seconds.
-- Epoch 3
Norm: 1.34, NNZs: 24, Bias: -0.075189, T: 222585390, Avg. loss: 0.644416, Objective: 0.649195
Total training time: 167.03 seconds.
-- Epoch 4
Norm: 1.27, NNZs: 23, Bias: -0.071363, T: 296780520, Avg. loss: 0.643980, Objective: 0.647844
Total training time: 222.53 seconds.
-- Epoch 5
Norm: 1.27, NNZs: 23, Bias: -0.076597, T: 370975650, Avg. loss: 0.644039, Objective: 0.647667
Total training time: 277.83 seconds.
-- Epoch 6
Norm: 1.27, NNZs: 25, Bias: -0.075675, T: 445170780, Avg. loss: 0.643932, Objective: 0.647585
Total training time: 333.01 seconds.
-- Epoch 7
Norm: 1.28, NNZs: 24, Bias: -0.076239, T: 519365910, Avg. loss: 0.643856, Objective: 0.647584
Total training time: 388.30 seconds.
-- Epoch

2026-04-17 20:26:23,468 [INFO] SGD done in 464.7 s


Norm: 1.31, NNZs: 25, Bias: -0.084418, T: 593561040, Avg. loss: 0.643840, Objective: 0.647625
Total training time: 443.47 seconds.
Convergence after 8 epochs took 443.47 seconds


2026-04-17 20:26:27,091 [INFO] SGD Classifier          AUC-ROC=0.6889  PR-AUC=0.0307  Brier=0.21127


In [ ]:

model_dir = os.path.join(out_path, "saved_models")
save_model(sgd_model, model_dir, "sgd")

#### Random Forest

In [ ]:
def grid_search_random_forest(X_train, y_train, weight_ratio, out_path,
                               sample_frac=0.05, sample_n=None, n_iter=10):
    import optuna
    from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
    from sklearn.metrics import roc_auc_score
    from sklearn.ensemble import RandomForestClassifier
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    n_total = len(y_train)
    if n_total == 0:
        raise ValueError("X_train / y_train are empty")
    y_train = np.asarray(y_train).astype(np.int32)

    # --- sample ---
    if sample_n is not None:
        n_sample = min(sample_n, n_total)
    else:
        n_sample = max(1, int(n_total * sample_frac))

    if n_sample < n_total:
        sss = StratifiedShuffleSplit(n_splits=1, train_size=n_sample, random_state=SEED)
        idx, _ = next(sss.split(np.zeros(n_total), y_train))
        X_gs, y_gs = X_train[idx], y_train[idx]
    else:
        X_gs, y_gs = X_train, y_train

    if isinstance(X_gs, np.ndarray):
        X_gs = X_gs.astype(np.float32, copy=False)

    sample_weights = np.where(y_gs == 1, weight_ratio, 1.0).astype(np.float32)
    log.info("RF search sample: %d / %d (%.2f%%) | pos rate: %.4f",
             len(y_gs), n_total, 100 * len(y_gs) / n_total, y_gs.mean())

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

    def objective(trial):
        params = dict(
            n_estimators     = trial.suggest_int("n_estimators", 100, 500, step=100),
            max_depth        = trial.suggest_int("max_depth", 3, 15),
            min_samples_leaf = trial.suggest_int("min_samples_leaf", 10, 200, log=True),
            min_samples_split= trial.suggest_int("min_samples_split", 2, 50),
            max_features     = trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3, 0.5]),
            max_samples      = trial.suggest_float("max_samples", 0.5, 1.0),  # row subsampling
            class_weight     = "balanced",
            random_state     = SEED,
            n_jobs           = 1,   # n_jobs=-1 только на уровне study.optimize
        )

        scores = []
        for train_idx, val_idx in cv.split(X_gs, y_gs):
            X_tr, X_val = X_gs[train_idx], X_gs[val_idx]
            y_tr, y_val = y_gs[train_idx], y_gs[val_idx]
            sw_tr = sample_weights[train_idx]

            model = RandomForestClassifier(**params)
            model.fit(X_tr, y_tr, sample_weight=sw_tr)
            scores.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))

        return np.mean(scores)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(objective, n_trials=n_iter, n_jobs=-1, show_progress_bar=True)

    log.info("Best RF: %s  AUC=%.4f", study.best_params, study.best_value)
    _save_sklearn_grid_results("RF_optuna", study, out_path)
    return study

grid_search_random_forest(X_train, y_train, weight_ratio, out_path)

### Plots and results

In [6]:
# ["lr", "rf", "sgd", "xgb", "lgb"]
model_dir = os.path.join(out_path, "saved_models")

lr_model = load_model(os.path.join(model_dir, "lr.pkl"))
rf_model = load_model(os.path.join(model_dir, "rf.pkl"))
sgd_model = load_model(os.path.join(model_dir, "sgd.pkl"))
xgb_model = load_model(os.path.join(model_dir, "xgb.pkl"))
lgb_model = load_model(os.path.join(model_dir, "lgb.pkl"))

In [18]:
# ── Plots ─────────────────────────────────────────────────────────────────
plot_lr_coefficients(lr_model, feature_names, out_path)
plot_feature_importance(rf_model, feature_names, "Random Forest", out_path)
plot_feature_importance(xgb_model, feature_names, "XGBoost", out_path)
plot_feature_importance(lgb_model, feature_names, "LightGBM", out_path)
plot_kaplan_meier(test_pd, out_path)

In [19]:
# ── Phase 3: save ─────────────────────────────────────────────────────────
log.info("=" * 60)
log.info("PHASE 3: Saving models + sklearn preprocessor")
log.info("=" * 60)

save_all(out_path, preprocessor=preprocessor, feature_names=feature_names,
         lr_model=lr_model, rf_model=rf_model, sgd_model=sgd_model,
         xgb_model=xgb_model, lgb_model=lgb_model,
         weight_ratio=weight_ratio, col_meta=col_meta)

2026-04-16 22:41:56,511 [INFO] ============================================================
2026-04-16 22:41:56,512 [INFO] PHASE 3: Saving models + sklearn preprocessor
2026-04-16 22:41:56,512 [INFO] ============================================================
2026-04-16 22:41:56,514 [INFO] Saved sklearn preprocessor → /home/yc-user/models/saved_models/preprocessor.pkl
2026-04-16 22:41:56,514 [INFO] Saved LR → /home/yc-user/models/saved_models/lr.pkl
2026-04-16 22:41:56,588 [INFO] Saved RF → /home/yc-user/models/saved_models/rf.pkl
2026-04-16 22:41:56,589 [INFO] Saved SGD → /home/yc-user/models/saved_models/sgd.pkl
2026-04-16 22:41:56,595 [INFO] Saved XGB → /home/yc-user/models/saved_models/xgb.pkl
2026-04-16 22:41:56,615 [INFO] Saved LGB → /home/yc-user/models/saved_models/lgb.pkl
2026-04-16 22:41:56,616 [INFO] Saved metadata.json


In [20]:
# ── Results table ─────────────────────────────────────────────────────────
all_m = [m_lr, m_rf, m_sgd, m_xgb, m_lgb]
results_df = pd.DataFrame(all_m).sort_values("AUC-ROC", ascending=False).reset_index(drop=True)

print("\n" + "="*65)
print(f"  BASE MODEL COMPARISON (split at {TRAIN_CUTOFF}, sample={sample_frac:.0%})")
print("="*65)
print(results_df.to_string(index=False))
print("="*65 + "\n")
results_df.to_csv(os.path.join(out_path, "model_comparison.csv"), index=False)


  BASE MODEL COMPARISON (split at 2014-10-31, sample=100%)
              Model  AUC-ROC  PR-AUC   Brier
            XGBoost   0.7225  0.0446 0.19972
           LightGBM   0.7221  0.0450 0.20204
      Random Forest   0.7117  0.0496 0.20538
Logistic Regression   0.6940  0.0314 0.21194
     SGD Classifier   0.6920  0.0307 0.21161



In [21]:
curve_data = [("LR", y_test, lr_p), ("RF", y_test, rf_p), ("SGD", y_test, sgd_p),
               ("XGBoost", y_test, xgb_p), ("LightGBM", y_test, lgb_p)]
plot_roc_curves(curve_data, out_path)
plot_pr_curves(curve_data, out_path)
plot_calibration(curve_data, out_path)
plot_smm_cpr(test_pd, curve_data, out_path)
plot_model_comparison(results_df, out_path)

## Improvements

In [19]:
import argparse
import json
import logging
import os
import pickle
import time

import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_curve,
)

from models.train_models import (
    SEED, load_raw_data, prepare_arrays,
    compute_weight_ratio, evaluate, load_model,
    plot_roc_curves, plot_pr_curves, plot_calibration,
    plot_model_comparison,
)

from catboost import CatBoostClassifier
import xgboost as xgb
import lightgbm as lgb

try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)


# ============================================================================
#  SMOTE OVERSAMPLING
# ============================================================================

def apply_smote(X_train, y_train, random_state=SEED):
    """
    Apply SMOTE (Synthetic Minority Oversampling Technique).

    Generates synthetic minority-class examples by interpolating between
    existing minority samples and their k nearest neighbours.

    Effect on AUC: typically +0.5–2 pp on imbalanced datasets.
    RAM: augmented dataset is ~2× the original for a 5% minority class.
    Requires: pip install imbalanced-learn
    """
    if not HAS_SMOTE:
        log.warning("imbalanced-learn not installed — skipping SMOTE")
        return X_train, y_train

    log.info("Applying SMOTE …")
    n_pos_before = np.sum(y_train == 1)
    sm = SMOTE(random_state=random_state, n_jobs=-1)
    X_res, y_res = sm.fit_resample(X_train, y_train)
    log.info("SMOTE: %d → %d positives (total %d → %d rows)",
             n_pos_before, np.sum(y_res == 1), len(y_train), len(y_res))
    return X_res, y_res


# ============================================================================
#  STACKING ENSEMBLE
# ============================================================================

def train_stacking_ensemble(X_train, y_train, X_test, weight_ratio):
    """
    Stacking ensemble: LR + RF + XGBoost + LightGBM → LR meta-learner.

    The meta-learner sees predicted probabilities from all base models
    and learns which model to trust in which region of feature space.
    Typically adds +1–3 pp AUC over the best single model.

    RAM: fits each base model 5× (cv=5) plus one final fit ≈ 6× a single fit.
    """
    log.info("Training stacking ensemble …")
    t0 = time.time()

    estimators = [
        ("sgd", SGDClassifier(
            loss="log_loss", penalty="l2", alpha=1e-4,
            max_iter=1000, tol=1e-3, class_weight="balanced",
            random_state=SEED, n_jobs=-1, verbose=0)),
        ("xgb", xgb.XGBClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=weight_ratio, tree_method="hist",
            random_state=SEED, n_jobs=-1, verbosity=0)),
        ("lgb", lgb.LGBMClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            num_leaves=31, subsample=0.8, colsample_bytree=0.8,
            boosting_type="gbdt", random_state=SEED, n_jobs=-1, verbosity=-1)),
       
    ]

    stack = StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(C=1.0, solver="lbfgs",
                                           max_iter=500, random_state=SEED),
        cv=3, stack_method="predict_proba", passthrough=False, n_jobs=-1)
    stack.fit(X_train, y_train)
    log.info("Stacking done in %.1f s", time.time() - t0)
    return stack, stack.predict_proba(X_test)[:, 1]


def train_stacking_ensemble_v2(X_train, y_train, X_test, weight_ratio):
    """
    Stacking ensemble: LR + RF + XGBoost + LightGBM → LR meta-learner.

    The meta-learner sees predicted probabilities from all base models
    and learns which model to trust in which region of feature space.
    Typically adds +1–3 pp AUC over the best single model.

    RAM: fits each base model 5× (cv=5) plus one final fit ≈ 6× a single fit.
    """
    log.info("Training stacking ensemble …")
    t0 = time.time()

    estimators = [
        ("sgd", SGDClassifier(
            loss="log_loss", penalty="l2", alpha=1e-4,
            max_iter=1000, tol=1e-3, class_weight="balanced",
            random_state=SEED, n_jobs=-1, verbose=0)),
        ("xgb", xgb.XGBClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=weight_ratio, tree_method="hist",
            random_state=SEED, n_jobs=-1, verbosity=0)),
        ("cat", CatBoostClassifier(
            iterations=200,
            depth=6,
            learning_rate=0.05,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=SEED,
            verbose=0,
            thread_count=-1
        )),
       
    ]

    stack = StackingClassifier(
        estimators=estimators,
        final_estimator=lgb.LGBMClassifier(
            n_estimators=50,
            max_depth=2,
            num_leaves=3,
            learning_rate=0.05,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1
        ),
        cv=3,
        stack_method="predict_proba",
        passthrough=True,
        n_jobs=-1
    )
    stack.fit(X_train, y_train)
    log.info("Stacking done in %.1f s", time.time() - t0)
    return stack, stack.predict_proba(X_test)[:, 1]



# ============================================================================
#  SOFT VOTING ENSEMBLE
# ============================================================================

def train_voting_ensemble(X_test, fitted_models):
    """
    Soft voting: average predicted probabilities from pre-fitted models.
    No additional training. Reduces variance → +0.5–1 pp AUC.
    """
    log.info("Building soft voting ensemble (%d models) …", len(fitted_models))
    probs = []
    for name, model in fitted_models.items():
        if model is None:
            continue
        try:
            p = model.predict_proba(X_test)[:, 1]
            probs.append(p)
            log.info("  %s → mean=%.4f", name, p.mean())
        except Exception as e:
            log.warning("  %s failed: %s", name, e)

    if not probs:
        return None
    return np.mean(probs, axis=0)


from scipy.stats import rankdata

def train_voting_ensemble_v2(X_test, fitted_models, weights=None, use_ranks=False):
    """
    Weighted soft voting (or rank voting) over pre-fitted models.
    """
    log.info("Building voting ensemble (%d models) …", len(fitted_models))

    preds = {}
    for name, model in fitted_models.items():
        if model is None:
            continue
        try:
            p = model.predict_proba(X_test)[:, 1]
            preds[name] = p
            log.info("  %s → mean=%.4f", name, p.mean())
        except Exception as e:
            log.warning("  %s failed: %s", name, e)

    if not preds:
        return None

    if weights is None:
        weights = {name: 1.0 for name in preds}

    total_weight = sum(weights.get(name, 0.0) for name in preds)
    if total_weight <= 0:
        raise ValueError("Sum of weights must be positive.")

    if use_ranks:
        from scipy.stats import rankdata
        ensemble = sum(
            weights.get(name, 0.0) * (rankdata(preds[name]) / len(preds[name]))
            for name in preds
        ) / total_weight
    else:
        ensemble = sum(
            weights.get(name, 0.0) * preds[name]
            for name in preds
        ) / total_weight

    return ensemble

# ============================================================================
#  ISOTONIC CALIBRATION
# ============================================================================

def calibrate_model(model, X_train, y_train, X_test, method="isotonic"):
    """
    Post-hoc probability calibration via CalibratedClassifierCV.

    Isotonic calibration learns a non-parametric monotone mapping that
    fixes miscalibrated probabilities. Improves AUC by +0.1–0.5 pp
    and PR-AUC by +0.5–2 pp.
    """
    log.info("Calibrating with %s regression …", method)
    cal = CalibratedClassifierCV(model, cv=5, method=method)
    cal.fit(X_train, y_train)
    return cal, cal.predict_proba(X_test)[:, 1]


# ============================================================================
#  OPTIMAL THRESHOLD (YOUDEN'S J)
# ============================================================================

def find_optimal_threshold(y_true, y_prob):
    """
    Find optimal classification threshold via Youden's J = TPR - FPR.
    Useful for OAS binary prepay/no-prepay decisions.
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    j = tpr - fpr
    idx = np.argmax(j)
    thresh = float(thresholds[idx])
    log.info("Optimal threshold: %.4f (J=%.4f, TPR=%.4f, FPR=%.4f)",
             thresh, j[idx], tpr[idx], fpr[idx])
    return thresh, float(j[idx])




In [5]:
model_dir = os.path.join(out_path, "saved_models")
sample_frac = 0.5
# ── Load preprocessor and data ────────────────────────────────────────────
preprocessor = load_model(os.path.join(model_dir, "preprocessor.pkl"))
train_pd, test_pd, col_meta = load_raw_data(out_path, sample_frac)
X_train, y_train, X_test, y_test = prepare_arrays(train_pd, test_pd, preprocessor)
weight_ratio = compute_weight_ratio(y_train)

2026-04-17 09:11:55,398 [INFO] Loading raw train Parquet …
2026-04-17 09:12:08,932 [INFO] Loading raw test Parquet …
2026-04-17 09:12:10,660 [INFO] Full sizes: train=74,195,130, test=6,824,737
2026-04-17 09:12:43,250 [INFO] After sampling (50%): train=37,097,565, test=3,412,368
2026-04-17 09:12:43,251 [INFO] Transforming train set …
2026-04-17 09:13:50,271 [INFO] Transforming test set …
2026-04-17 09:13:56,608 [INFO] Feature matrix: train=(37097565, 70), test=(3412368, 70) (11.3 GB total)
2026-04-17 09:13:56,649 [INFO] Class balance: n_neg=36604221, n_pos=493344, weight_ratio=74.20


In [6]:
# ── Load base models ──────────────────────────────────────────────────────
models = {}
for name in ["lr", "rf", "sgd", "xgb", "lgb"]:
    p = os.path.join(model_dir, f"{name}.pkl")
    if os.path.exists(p):
        models[name] = load_model(p)

In [10]:
# ── Base model evaluation (for comparison) ────────────────────────────────
base_metrics = []
base_curves = []
preds = {}
for name, model in models.items():
    if model is None:
        continue
    probs = model.predict_proba(X_test)[:, 1]
    preds[name] = probs
    m = evaluate(y_test, probs, name.upper())
    base_metrics.append(m)
    base_curves.append((name.upper(), y_test, probs))

2026-04-17 09:48:24,670 [INFO] LR                      AUC-ROC=0.6941  PR-AUC=0.0316  Brier=0.21195
[Parallel(n_jobs=64)]: Using backend ThreadingBackend with 64 concurrent workers.
[Parallel(n_jobs=64)]: Done  72 tasks      | elapsed:    0.8s
[Parallel(n_jobs=64)]: Done 300 out of 300 | elapsed:    2.6s finished
2026-04-17 09:48:28,971 [INFO] RF                      AUC-ROC=0.7117  PR-AUC=0.0498  Brier=0.20535
2026-04-17 09:48:30,731 [INFO] SGD                     AUC-ROC=0.6919  PR-AUC=0.0308  Brier=0.21161
2026-04-17 09:48:32,788 [INFO] XGB                     AUC-ROC=0.7226  PR-AUC=0.0451  Brier=0.19969
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026-04-17 09:48:36,787 [INFO] LGB                     AUC-ROC=0.7222  PR-AUC=0.0455  Brier=0.20201


In [11]:
preds_df = pd.DataFrame(preds)
print(preds_df.corr())

           lr        rf       sgd       xgb       lgb
lr   1.000000  0.810993  0.985229  0.832580  0.836685
rf   0.810993  1.000000  0.790774  0.926538  0.934784
sgd  0.985229  0.790774  1.000000  0.822356  0.825498
xgb  0.832580  0.926538  0.822356  1.000000  0.989276
lgb  0.836685  0.934784  0.825498  0.989276  1.000000


In [8]:
# ── G. Stacking ───────────────────────────────────────────────────────────
stack_model, stack_p = train_stacking_ensemble(
    X_train, y_train, X_test, weight_ratio)
m_stack = evaluate(y_test, stack_p, "Stacking Ensemble")

2026-04-17 09:14:11,790 [INFO] Training stacking ensemble …
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026-04-17 09:39:13,145 [INFO] Stacking done in 1501.4 s
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026-04-17 09:39:15,923 [INFO] Stacking Ensemble       AUC-

In [14]:
# ── G. Stacking ver 2───────────────────────────────────────────────────────────
stack_model, stack_p = train_stacking_ensemble_v2(
    X_train, y_train, X_test, weight_ratio)
m_stack = evaluate(y_test, stack_p, "Stacking Ensemble")

2026-04-17 09:56:19,153 [INFO] Training stacking ensemble …
2026-04-17 10:22:51,557 [INFO] Stacking done in 1592.4 s
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026-04-17 10:22:53,585 [INFO] Stacking Ensemble       AUC-ROC=0.7174  PR-AUC=0.0436  Brier=0.01232


In [18]:
# ── H. Voting ─────────────────────────────────────────────────────────────
vote_p = train_voting_ensemble(X_test, models)
m_vote = evaluate(y_test, vote_p, "Voting Ensemble")

2026-04-17 10:30:50,105 [INFO] Building soft voting ensemble (5 models) …
2026-04-17 10:30:50,179 [INFO]   lr → mean=0.4390
[Parallel(n_jobs=64)]: Using backend ThreadingBackend with 64 concurrent workers.
[Parallel(n_jobs=64)]: Done  72 tasks      | elapsed:    0.8s
[Parallel(n_jobs=64)]: Done 300 out of 300 | elapsed:    2.6s finished
2026-04-17 10:30:52,844 [INFO]   rf → mean=0.4354
2026-04-17 10:30:52,935 [INFO]   sgd → mean=0.4399
2026-04-17 10:30:53,569 [INFO]   xgb → mean=0.4172
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026-04-17 10:30:55,955 [INFO]   lgb → mean=0.4223
2026-04-17 10:30:57,609 [INFO] Voting Ensemble         AUC-ROC=0.7170  PR-AUC=0.0456  Brier=0.20377


In [24]:
# ── H. Voting ver 2─────────────────────────────────────────────────────────────
weights = {
    "xgb": 0.5,
    "lgb": 0.35,
    "sgd": 0,
    "rf": 0.15,
    "lr": 0,
}

vote_p = train_voting_ensemble_v2(X_test, models, weights)
m_vote = evaluate(y_test, vote_p, "Voting Ensemble")

2026-04-17 10:36:56,949 [INFO] Building voting ensemble (5 models) …
2026-04-17 10:36:57,025 [INFO]   lr → mean=0.4390
[Parallel(n_jobs=64)]: Using backend ThreadingBackend with 64 concurrent workers.
[Parallel(n_jobs=64)]: Done  72 tasks      | elapsed:    0.8s
[Parallel(n_jobs=64)]: Done 300 out of 300 | elapsed:    2.6s finished
2026-04-17 10:36:59,727 [INFO]   rf → mean=0.4354
2026-04-17 10:36:59,823 [INFO]   sgd → mean=0.4399
2026-04-17 10:37:00,371 [INFO]   xgb → mean=0.4172
/home/yc-user/venvs/thesis/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026-04-17 10:37:02,727 [INFO]   lgb → mean=0.4223
2026-04-17 10:37:04,364 [INFO] Voting Ensemble         AUC-ROC=0.7227  PR-AUC=0.0483  Brier=0.20069


In [20]:
models

{'lr': LogisticRegression(C=1000.0, class_weight='balanced', max_iter=200, n_jobs=-1,
                    penalty='l2', random_state=42, solver='saga', verbose=1),
 'rf': RandomForestClassifier(class_weight='balanced', max_depth=10,
                        min_samples_leaf=50, n_estimators=300, n_jobs=-1,
                        random_state=42, verbose=1),
 'sgd': SGDClassifier(class_weight='balanced', loss='log_loss', n_jobs=-1,
               random_state=42, verbose=1),
 'xgb': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=0.8, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric='auc', feature_types=None,
               feature_weights=None, gamma=None, grow_policy=None,
               importance_type=None, interaction_constraints=None,
               learning_rate=0.05, max_bin=None, max_cat_threshold=None,
               max_cat_to

In [25]:
# ── I. Isotonic calibration (on best boosting model) ──────────────────────
best_name = "xgb"
cal_model, cal_p, m_cal = None, None, None
if best_name:
    cal_model, cal_p = calibrate_model(
        models[best_name], X_train, y_train, X_test)
    m_cal = evaluate(y_test, cal_p, f"{best_name.upper()} + Isotonic")

2026-04-17 10:37:15,776 [INFO] Calibrating with isotonic regression …
2026-04-17 10:53:30,093 [INFO] XGB + Isotonic          AUC-ROC=0.7226  PR-AUC=0.0450  Brier=0.01230


In [26]:
# ── Save ensemble models ──────────────────────────────────────────────────
for name, model in [("stacking", stack_model), ("calibrated", cal_model)]:
    if model is not None:
        p = os.path.join(model_dir, f"{name}.pkl")
        with open(p, "wb") as fh:
            pickle.dump(model, fh, protocol=pickle.HIGHEST_PROTOCOL)
        log.info("Saved %s → %s", name, p)

2026-04-17 10:53:30,202 [INFO] Saved stacking → /home/yc-user/models/saved_models/stacking.pkl
2026-04-17 10:53:30,229 [INFO] Saved calibrated → /home/yc-user/models/saved_models/calibrated.pkl
